# Clase 03 — Customer Journey: comportamiento a lo largo del tiempo

> **Pregunta de negocio:** ¿Cómo se comportan los clientes y dónde los
> perdemos?

Cambiamos de empresa. Ahora trabajamos para una tienda online de productos
variados. Su equipo de producto tiene una preocupación distinta a la de
marketing en las clases anteriores:

> "En cuatro meses han entrado a la web más de un millón de visitantes y
> compra una fracción mínima. Queremos saber en qué punto del recorrido se
> nos van, cuánto tardan en decidirse los que compran y si los que vienen una
> vez vuelven. Y queremos tres propuestas concretas de mejora para el próximo
> trimestre."

La tabla de cliente de la clase 1 no sirve para responder. Resume a cada
cliente en una fila (cuánto gastó, cuántas veces compró, cuándo fue la
última vez), pero no dice **en qué orden** hizo las cosas ni **cuánto
tiempo** pasó entre ellas. Para eso hay que mirar al cliente como una
**secuencia de eventos en el tiempo**: lo que en marketing y producto se
llama el ***customer journey*** (el recorrido del cliente).

Los datos son el log de eventos de **Retailrocket**, una tienda online real
que publicó cuatro meses y medio de actividad anonimizada (mayo a septiembre
de 2015): cada vez que un visitante ve un producto, lo añade al carrito o lo
compra, queda una fila.

### Antes de empezar: qué necesitas

1. **Python 3 con Jupyter** y las librerías `pandas`, `numpy`, `matplotlib`
   y `pyarrow` (sirve para leer los ficheros `.parquet`). Si te falta alguna,
   instálala desde una celda con `%pip install pyarrow` (cambiando el nombre
   por el de la que falte) y reinicia el kernel.
2. **Un fichero de datos**, que te dará el profesor. Cópialo a mano en **la
   misma carpeta que este notebook**, sin cambiarle el nombre:

| Fichero | Tamaño | Para qué sirve |
| --- | --- | --- |
| `events.parquet` | 37 MB | El log de eventos de Retailrocket: todo el laboratorio y las celdas de rescate |

La primera celda de código comprueba que el fichero está en la carpeta y, si
falta, te lo dice.

### Qué vas a aprender y qué vas a entregar

Al terminar habrás auditado un log de eventos, lo habrás dividido en
sesiones, habrás construido el funnel vista → carrito → compra por sesión y
por visitante, medido la retención por cohortes, estimado una matriz de
transición entre estados y formulado tres hipótesis de mejora priorizadas.

Entregables (en la carpeta `salidas/`, que se crea sola):

- `journey.csv`: el resumen cuantitativo del journey, cada cifra con su
  unidad y su umbral;
- `hipotesis.csv`: tus tres hipótesis de mejora, con observación, posible
  causa, acción, métrica, impacto, esfuerzo y prioridad.

### Índice

| Parte | Contenido |
| --- | --- |
| 0 | Del cliente-fila al cliente-secuencia y configuración |
| A | Demo guiada: sesionizar un visitante paso a paso |
| B1 | El event log (ejercicio 1) |
| B2 | Sesiones (ejercicios 2 y 3) |
| B3 | Funnels (ejercicios 4, 5 y 6) |
| B4 | Cohortes y retención (ejercicio 7) |
| B5 | Transiciones entre estados (ejercicios 8 y 9) |
| B6 | De la observación a la hipótesis (ejercicio 10) |
| C | Síntesis |
| Ampliación | Opcional (X1 y X2) |

### Cómo trabajar con este notebook

- Cada parte empieza con una **explicación** y un **ejemplo resuelto**:
  léelos y ejecuta sus celdas antes de pasar a los ejercicios. Todo lo que
  necesitas para resolverlos está en el notebook.
- Las celdas de ejercicio tienen huecos `COMPLETA` con un comentario
  `# TODO:` que dice qué falta. Si al ejecutar ves
  `NameError: name 'COMPLETA' is not defined`, te falta rellenar un hueco.
- Tras cada ejercicio de código hay una **celda de comprobación**: si
  imprime `✔ Correcto`, vas bien; si no, su mensaje te orienta.
- Las **celdas de rescate** preparan un resultado para que puedas seguir si
  te has atascado o te has saltado un ejercicio: cambia
  `USAR_RESCATE = False` por `True` y ejecútala.
- Las celdas **Tu respuesta** son para escribir tus conclusiones.
- Si ves un `FileNotFoundError` al cargar datos, el notebook se ha abierto
  fuera de su carpeta o falta el fichero: el mensaje dice en qué carpeta lo
  está buscando.
- El log tiene 2,8 millones de filas: la mayoría de las celdas tardan unos
  segundos, y alguna, más. Espera a que termine (el indicador `[*]` de la
  celda pasa a un número) antes de ejecutar la siguiente.

## 0. Del cliente-fila al cliente-secuencia

En la clase 1 pasamos de líneas de factura a una fila por cliente
**agregando**: sumas, recuentos, máximos. Agregar resume, pero destruye tres
cosas:

- **El orden.** "Vio, añadió al carrito, compró" y "compró, vio, añadió al
  carrito" dan la misma fila (1 vista, 1 carrito, 1 compra) y son
  comportamientos muy distintos.
- **Los tiempos.** Comprar tres minutos después de ver el producto o tres
  semanas después indica decisiones distintas.
- **Los recorridos.** Qué caminos siguen los visitantes, dónde se repiten,
  dónde se interrumpen.

El *customer journey* estudia justo eso que la agregación borra. El
notebook sigue este recorrido, y cada paso responde a una pregunta:

| Paso | Pregunta | Parte |
| --- | --- | --- |
| Log de eventos | ¿Qué datos tenemos y qué calidad tienen? | B1 |
| Sesiones | ¿Qué es una visita, si el log no lo dice? | A, B2 |
| Funnel | ¿En qué etapa se pierden los visitantes y cuánto tardan los que compran? | B3 |
| Cohortes | ¿Vuelven los que vienen una vez? | B4 |
| Transiciones | ¿Qué recorridos siguen y dónde se cortan? | B5 |
| Hipótesis | ¿Qué proponemos cambiar y cómo sabremos si funciona? | B6 |

En esta clase no hay ningún algoritmo de *machine learning*: solo ordenar,
contar y dividir. La dificultad está en decidir **qué** contar y **con qué
unidad**: un evento, una sesión o un visitante. La misma pregunta ("¿qué
proporción compra?") tiene respuestas distintas según la unidad, así que
cada cifra del notebook dirá la suya.

**El cierre de cada análisis**, como en la clase 1:

**métrica técnica → interpretación → decisión → limitación → guardrail**

En las preguntas de interpretación bastará con los dos primeros pasos: cita
una cifra (con su unidad) y di qué decisión permite tomar.

### Configuración

Importamos las librerías, comprobamos que el fichero de la clase está en
esta carpeta y fijamos un estilo sobrio para las figuras. Aquí no hace falta
semilla: nada de lo que haremos usa números al azar. La función
`linea_de_tiempo` solo dibuja; no hace falta entenderla.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow  # no se usa directamente: pandas la necesita para leer los .parquet

# El fichero de datos tiene que estar en la misma carpeta que este notebook.
FICHEROS_CLASE = ["events.parquet"]
faltan = []
for fichero in FICHEROS_CLASE:
    if not Path(fichero).exists():
        faltan.append(fichero)
if faltan:
    raise FileNotFoundError(f"Faltan estos ficheros en la carpeta {Path.cwd()}: {faltan}. "
                            "Cópialos a mano junto al notebook, sin cambiarles el nombre "
                            "(ver 'Antes de empezar'). Si ya están, abre el notebook "
                            "desde su propia carpeta.")

# Aquí se guardan tus entregables.
DIR_SALIDAS = Path("salidas")
DIR_SALIDAS.mkdir(exist_ok=True)

pd.set_option("display.width", 100)
pd.set_option("display.max_columns", 12)
pd.set_option("display.float_format", "{:,.4f}".format)   # 4 decimales en todas las tablas

# Colores y estilo de las figuras (no hace falta entenderlo para seguir la clase).
AZUL, NARANJA, AQUA, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRIS,
    "xtick.color": GRIS,
    "ytick.color": GRIS,
    "axes.grid": True,
    "grid.color": "#e4e3df",
    "grid.linewidth": 0.8,
})
# Un color, una marca y una letra por tipo de evento: V vista, C carrito, T compra (transaction).
ESTILO = {"view": (AZUL, "o", "V"), "addtocart": (NARANJA, "s", "C"),
          "transaction": (AQUA, "D", "T")}


def linea_de_tiempo(ax, tiempos, eventos_tl, unidad):
    """Dibuja eventos sobre una línea de tiempo (función de dibujo)."""
    ax.axhline(0, color=GRIS, linewidth=1, zorder=1)
    for t, ev in zip(tiempos, eventos_tl):
        color, marca, letra = ESTILO[ev]
        ax.scatter(t, 0, s=130, color=color, marker=marca, edgecolors="white",
                   linewidths=1.2, zorder=3)
        ax.annotate(letra, (t, 0), xytext=(0, 10), textcoords="offset points",
                    ha="center", fontsize=10, fontweight="bold", color=color)
    ax.set(yticks=[], xlabel=unidad, ylim=(-1, 1.2))
    ax.spines["left"].set_visible(False)
    ax.grid(axis="y", visible=False)


print("✔ Configuración lista")

## Parte A · Demo guiada: sesionizar un visitante paso a paso

Un visitante puede entrar a la web el lunes por la mañana, volver el lunes
por la tarde y otra vez el jueves. Cada una de esas visitas es una
**sesión**: un periodo de actividad continua. Muchas preguntas de negocio
se hacen por sesión ("¿qué porcentaje de visitas acaba en compra?"), pero
muchos logs, como el de hoy, **no traen un identificador de sesión**: solo
el visitante y la hora de cada evento. Hay que construirlo.

La regla habitual es un **umbral de inactividad**: si entre dos eventos
consecutivos del mismo visitante pasan más de X minutos, empieza una sesión
nueva. Google Analytics, por ejemplo, usa 30 minutos por defecto, y es el
valor que tomaremos como punto de partida.

Lo hacemos con un visitante inventado que tiene ocho eventos en un día. La
tabla llega **desordenada**, como pasa a menudo cuando un log junta datos
de varios servidores:

In [ ]:
log_demo = pd.DataFrame({
    "visitor_id": 1,
    "timestamp": pd.to_datetime([
        "2015-06-01 10:50", "2015-06-01 10:00", "2015-06-01 14:00", "2015-06-01 10:05",
        "2015-06-01 14:03", "2015-06-01 10:12", "2015-06-01 11:10", "2015-06-01 11:02"]),
    "event": ["view", "view", "view", "view", "transaction", "addtocart", "addtocart", "view"],
})
log_demo

**A mano.** Ordenados, los eventos son 10:00, 10:05, 10:12, 10:50, 11:02,
11:10, 14:00 y 14:03. Los minutos que pasan entre cada uno y el anterior son
5, 7, **38**, 12, 8, **170** y 3. Con un umbral de 30 minutos hay dos huecos
que lo superan (entre las 10:12 y las 10:50, y entre las 11:10 y las 14:00),
así que el día se divide en **tres sesiones**: de 10:00 a 10:12, de 10:50 a
11:10 y de 14:00 a 14:03. Ahora lo mismo con pandas, en tres pasos.

**Paso 1: ordenar** por visitante y por tiempo. Sin este paso, las
diferencias de tiempo del paso 2 restarían eventos que no son consecutivos
y no significarían nada. `reset_index(drop=True)` vuelve a numerar las filas
0, 1, 2… en el nuevo orden, para que se lean mejor.

In [ ]:
log_demo = log_demo.sort_values(["visitor_id", "timestamp"]).reset_index(drop=True)
log_demo

**Paso 2: el hueco** de cada evento con el evento anterior **del mismo
visitante**. `groupby("visitor_id")["timestamp"].diff()` resta a cada hora
la anterior dentro de cada visitante (aquí solo hay uno, pero con el log
real habrá 1,4 millones). El primer evento de cada visitante no tiene
anterior: su hueco es nulo (`NaT`, "no es una fecha"). Para leerlo en
minutos, `dt.total_seconds()` pasa la duración a segundos y dividimos
entre 60.

In [ ]:
hueco_demo = log_demo.groupby("visitor_id")["timestamp"].diff()
log_demo["hueco_min"] = hueco_demo.dt.total_seconds() / 60
log_demo

**Paso 3: marcar los inicios de sesión y numerarlos.** Un evento abre
sesión si su hueco es nulo (es el primero del visitante: `isna()`) **o**
(`|`) si es mayor que el umbral. `pd.Timedelta(minutes=30)` es una duración
de 30 minutos que se puede comparar con los huecos. La **suma acumulada**
(`cumsum`) de esas marcas numera las sesiones: `True` cuenta como 1 y
`False` como 0, así que el contador sube en uno cada vez que aparece un
inicio.

In [ ]:
nueva_demo = hueco_demo.isna() | (hueco_demo > pd.Timedelta(minutes=30))
log_demo["inicio"] = nueva_demo
log_demo["session_id"] = nueva_demo.cumsum()
log_demo

La columna `inicio` vale `True` en las filas 0, 3 y 6, y su suma acumulada
es 1, 1, 1, 2, 2, 2, 3, 3: las tres sesiones del cálculo a mano.

**¿Y si el umbral fuera de 60 minutos?** Solo cambia la comparación:

In [ ]:
for umbral in [30, 60]:
    nueva_umbral = hueco_demo.isna() | (hueco_demo > pd.Timedelta(minutes=umbral))
    print(f"Umbral {umbral} min -> {nueva_umbral.cumsum().nunique()} sesiones")

Dibujamos el día del visitante con los dos umbrales. Para que se lean todos
los huecos, los eventos aparecen equiespaciados y el tiempo real entre cada
uno y el anterior va rotulado debajo de la línea; los huecos que superan el
umbral, en rojo, son los que cortan la sesión.

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
# Los eventos se dibujan equiespaciados; cada hueco lleva rotulados sus minutos.
fig, axes = plt.subplots(2, 1, figsize=(8.5, 4.2), sharex=True)
posicion = np.arange(len(log_demo))
huecos_log = log_demo["hueco_min"].to_numpy()
for ax, umbral in zip(axes, [30, 60]):
    sesion = (hueco_demo.isna() | (hueco_demo > pd.Timedelta(minutes=umbral))).cumsum().to_numpy()
    for k, ses in enumerate(np.unique(sesion)):
        tramo = posicion[sesion == ses]
        ax.axvspan(tramo.min() - 0.35, tramo.max() + 0.35, color=[AZUL, NARANJA, AQUA][k],
                   alpha=0.15, zorder=0)
        ax.text(tramo.mean(), -0.75, f"sesión {ses}", ha="center", fontsize=9)
    linea_de_tiempo(ax, posicion, log_demo["event"], "")
    for i in range(1, len(log_demo)):
        corta = huecos_log[i] > umbral
        ax.text(i - 0.5, -0.35, f"{huecos_log[i]:.0f}'", ha="center", fontsize=9, zorder=5,
                color="#b3261e" if corta else GRIS, fontweight="bold" if corta else "normal",
                bbox=dict(facecolor="white", edgecolor="none", pad=1) if corta else None)
        if corta:
            ax.axvline(i - 0.5, color="#b3261e", linewidth=1.5, linestyle="--")
    ax.set_title(f"Umbral de {umbral} minutos: {len(np.unique(sesion))} sesiones "
                 "(en rojo, los huecos que cortan)", fontsize=10)
axes[1].set_xticks(posicion, log_demo["timestamp"].dt.strftime("%H:%M"))
axes[1].set_xlabel("hora del evento (bajo la línea, minutos desde el evento anterior)")
fig.tight_layout()
plt.show()

Con 60 minutos, el hueco de 38 minutos ya no corta: la mañana es una sola
sesión y el día tiene dos. ¿Cuál es la respuesta "verdadera"? Ninguna.
¿Es la misma visita alguien que deja la pestaña abierta 38 minutos mientras
atiende una llamada y luego sigue mirando? Depende de para qué queramos la
sesión. **El umbral es una decisión**, y como toda decisión conviene
documentarla y comprobar cuánto cambia los resultados. Lo harás con el log
real en B2.

**Resumen de la demo:** ordenar → hueco con el anterior del mismo visitante
→ inicio si el hueco es nulo o mayor que el umbral → suma acumulada. En el
ejercicio 2 convertirás estos tres pasos en una función.

## Parte B · Laboratorio

### B1. El event log

**Qué se pierde al agregar.** Dos visitantes inventados tienen exactamente
los mismos eventos: 3 vistas, 1 carrito y 1 compra. El primero lo hace todo
en 6 minutos; el segundo entra cinco días distintos y compra a las tres
semanas. Los escribimos como un log y los agregamos con `pd.crosstab`, que
cuenta cuántas filas hay de cada combinación de dos columnas (aquí,
visitante × tipo de evento):

In [ ]:
inicio_ejemplo = pd.Timestamp("2015-06-01 10:00")
dos_visitantes_ejemplo = pd.DataFrame({
    "visitor_id": [1] * 5 + [2] * 5,
    # pd.to_timedelta convierte números en duraciones: minutos para el 1, días para el 2.
    "timestamp": list(inicio_ejemplo + pd.to_timedelta([0, 1.5, 2.5, 3.5, 6], unit="min"))
                 + list(inicio_ejemplo + pd.to_timedelta([0, 4, 9, 15, 20], unit="D")),
    "event": ["view", "view", "view", "addtocart", "transaction"] * 2,
})
pd.crosstab(dos_visitantes_ejemplo["visitor_id"], dos_visitantes_ejemplo["event"])

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
# Dos visitantes con la misma fila agregada (3 vistas, 1 carrito, 1 compra).
fig, axes = plt.subplots(2, 1, figsize=(8, 3.6))
linea_de_tiempo(axes[0], [0, 1.5, 2.5, 3.5, 6],
                ["view", "view", "view", "addtocart", "transaction"], "minutos")
axes[0].set(title="Visitante 1: decide en una sola visita de 6 minutos", xlim=(-0.5, 7))
linea_de_tiempo(axes[1], [0, 4, 9, 15, 20],
                ["view", "view", "view", "addtocart", "transaction"], "días")
axes[1].set(title="Visitante 2: entra cinco días distintos y compra a las tres semanas",
            xlim=(-0.8, 22))
fig.suptitle("Misma fila agregada (3 vistas, 1 carrito, 1 compra), dos recorridos distintos",
             fontsize=11)
fig.tight_layout()
plt.show()

Para una tabla de cliente como la de la clase 1 son el mismo cliente. Pero
el primero llegó con una intención clara y la resolvió en una visita; el
segundo dudó tres semanas y volvió cinco veces. Al primero le sobra
cualquier recordatorio; al segundo, quizá un recordatorio a tiempo le habría
hecho comprar antes. Las decisiones de producto dependen justo de lo que la
fila no guarda.

**Estructura de un event log.** Un *event log* (registro de eventos) es una
tabla con una fila por cosa que ha ocurrido. Casi siempre tiene cuatro
elementos:

| Elemento | Pregunta | En Retailrocket |
| --- | --- | --- |
| Usuario | ¿Quién? | `visitor_id` |
| Marca temporal | ¿Cuándo? | `timestamp` |
| Tipo de evento | ¿Qué hizo? | `event`: `view`, `addtocart`, `transaction` |
| Objeto | ¿Sobre qué? | `item_id` (y `transaction_id` en las compras, que no usaremos) |

Es el modelo cliente–evento–producto de la clase 1, pero sin agregar. Ojo:
cada fila `transaction` es **un producto comprado**, no un pedido: un pedido
de tres productos deja tres filas.

**Antes de analizar, la calidad.** Como en la clase 1, lo primero es
auditar: tamaño, periodo, duplicados exactos (mismo visitante, evento,
producto y milisegundo) y cuántos eventos tiene cada visitante. Esta última
cifra suele tener una **cola larga** extrema, como los productos de la
clase 1: muchísimos visitantes con uno o dos eventos y unos pocos con miles.
Un visitante con miles de eventos puede ser un robot que recorre la web,
una herramienta interna o un comprador profesional; no lo sabremos solo con
el log, pero hay que tenerlo localizado, porque en una media por visitante
puede pesar más que miles de visitantes normales. Para ver una cola así se
usa un gráfico con los **dos ejes en escala logarítmica** (lo verás después
del ejercicio).

**Identidad.** `visitor_id` no es un cliente identificado como el
`customer_id` de la clase 1, sino un **visitante anónimo**, normalmente una
cookie del navegador. La misma persona en el móvil y en el portátil son dos
visitantes; alguien que borra las cookies se convierte en un visitante
nuevo. Todo lo que midamos "por visitante" hereda esa limitación.

### Ejercicio 1 · Auditar el log de eventos · completar código + interpretar

Pistas:

```python
# pd.read_parquet("fichero.parquet") carga un fichero parquet en una tabla.
# tabla.duplicated() marca True las filas idénticas a una anterior; .sum() las cuenta.
# tabla["columna"].nunique() cuenta valores distintos.
# tabla["columna"].value_counts() cuenta cuántas filas hay de cada valor.
# tabla.groupby("visitor_id").size() cuenta filas por visitante.
# serie.describe(percentiles=[0.5, 0.9, 0.99]) da recuento, media, mínimo, máximo y esos percentiles.
```

(a) Completa la carga de `events.parquet`, los visitantes y productos
distintos, los duplicados exactos y el recuento por tipo de evento.

In [ ]:
# TU CÓDIGO
eventos = COMPLETA  # TODO: carga el fichero events.parquet
print(eventos.shape)
print("Periodo:", eventos["timestamp"].min(), "->", eventos["timestamp"].max())
n_visitantes = COMPLETA  # TODO: número de visitantes distintos
n_productos = COMPLETA  # TODO: número de productos distintos
n_duplicados = COMPLETA  # TODO: número de filas duplicadas exactas
print(f"Visitantes: {n_visitantes:,}")
print(f"Productos:  {n_productos:,}")
print(f"Duplicados exactos: {n_duplicados:,}")
recuento = COMPLETA  # TODO: eventos de cada tipo
recuento

In [ ]:
# Comprobación
assert eventos.shape == (2756101, 5), "Revisa la carga: el log tiene 2 756 101 filas y 5 columnas."
assert n_duplicados == 460, "Revisa los duplicados: cuéntalos antes de eliminarlos (son 460)."
assert n_visitantes == 1_407_580, "Revisa los visitantes: cuenta valores distintos de visitor_id."
print("✔ Correcto")

(b) Completa `por_visitante` (eventos de cada visitante) y su resumen con
los percentiles 50, 90 y 99.

In [ ]:
# TU CÓDIGO
por_visitante = COMPLETA  # TODO: eventos de cada visitante
resumen_pv = COMPLETA  # TODO: resumen con percentiles 50, 90 y 99
print(f"visitantes: {int(resumen_pv['count']):,}")   # el recuento, como número entero
# El resto con un decimal (sin float_format se verían los 4 decimales fijados en la configuración).
print(resumen_pv.drop("count").to_string(float_format="{:,.1f}".format))

In [ ]:
# Comprobación
assert len(por_visitante) == 1_407_580, "por_visitante debe tener una fila por visitante."
assert por_visitante.max() == 7757, "Revisa por_visitante: cuenta filas por visitante con groupby(...).size()."
print("✔ Correcto")

Los tres visitantes con más eventos, cuánto pesan en el total y cuántos
visitantes tienen un solo evento. Después, la figura: cuántos visitantes hay
con 1 evento, con 2, con 3…, con los dos ejes en escala logarítmica.

In [ ]:
top3 = por_visitante.sort_values(ascending=False).head(3)   # los tres con más eventos
print(top3.to_string())
print(f"Los tres suman {top3.sum():,} eventos: el {top3.sum() / len(eventos):.2%} del log")
print(f"Visitantes con un solo evento: {(por_visitante == 1).sum():,} "
      f"({(por_visitante == 1).mean():.1%})")

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
fig, ax = plt.subplots(figsize=(7, 3.4))
frecuencia = por_visitante.value_counts().sort_index()
ax.scatter(frecuencia.index, frecuencia.to_numpy(), s=12, color=AZUL)
ax.set(xscale="log", yscale="log", xlabel="eventos del visitante (escala log)",
       ylabel="número de visitantes (escala log)",
       title="¿Cuántos eventos tiene cada visitante?")
ax.annotate(f"{frecuencia.loc[1]:,} visitantes\ncon un solo evento".replace(",", " "),
            (1, frecuencia.loc[1]), xytext=(8, -28), textcoords="offset points", fontsize=9)
ax.annotate(f"un visitante con\n{por_visitante.max():,} eventos".replace(",", " "),
            (por_visitante.max(), 1), xytext=(-95, 25), textcoords="offset points",
            fontsize=9, arrowprops=dict(arrowstyle="->", color=GRIS))
fig.tight_layout()
plt.show()

Los puntos bajan casi en línea recta: por cada visitante con muchos eventos
hay muchísimos con pocos. A la derecha, por encima de unos 100 eventos, los
puntos se quedan en 1: son visitantes únicos, cada uno con su propio número
de eventos, y entre ellos están los casos raros.

Por último, eliminamos los 460 duplicados exactos, como en la clase 1. A
partir de aquí, `eventos` es el log limpio:

In [ ]:
eventos = eventos.drop_duplicates()
print(f"Eventos sin duplicados: {len(eventos):,}")

Preguntas: (i) ¿cuántos carritos y cuántas compras hay por cada 100 vistas?
¿Qué adelanta del problema de negocio? (ii) El visitante 1150086 tiene
7 757 eventos. ¿Qué puede ser, qué harías con él antes de calcular medias
por visitante y qué cambia que `visitor_id` sea una cookie?

**Tu respuesta:**

### B2. Sesiones

La celda siguiente vuelve a cargar el log y elimina los duplicados, para que
esta parte funcione aunque el ejercicio 1 se haya quedado a medias:

In [ ]:
eventos = pd.read_parquet("events.parquet").drop_duplicates()
print(f"Eventos sin duplicados: {len(eventos):,}")

**Del procedimiento a una función.** En la demo sesionizamos un visitante
en tres pasos. Con el log real los repetiremos varias veces, con distintos
umbrales, así que conviene escribirlos una vez como **función**: un bloque
de código con nombre que recibe datos (`eventos`, `umbral_min`) y devuelve
un resultado con `return`. Los pasos son los de la demo:

1. **Ordenar** por visitante y tiempo (`sort_values`). Añadimos `.copy()`
   para trabajar sobre una copia y no tocar la tabla que recibe la función.
2. **Hueco** con el evento anterior del mismo visitante
   (`groupby("visitor_id")["timestamp"].diff()`).
3. **Inicio de sesión** si el hueco es nulo o mayor que el umbral
   (`pd.Timedelta(minutes=umbral_min)`), y numeración con `cumsum`.

El orden importa: sin el paso 1, el paso 2 restaría a cada evento el que
esté encima en la tabla, aunque sea de otro día o de antes en el tiempo.

**El umbral es una decisión.** Ninguna cifra por sesión es "la verdadera":
el número de sesiones, los eventos por sesión o la proporción de sesiones
con compra dependen del umbral. Por eso cada una se cita con su umbral, y
conviene comprobar cuánto cambian con otros valores (**análisis de
sensibilidad**).

**¿Hay un umbral que sugieran los propios datos?** Los huecos entre eventos
consecutivos de un visitante son de dos tipos: los de **dentro** de una
visita (el tiempo de mirar un producto y pasar al siguiente, segundos o
minutos) y los de **entre** visitas (el visitante vuelve al día siguiente o
días después). Si dibujamos todos los huecos, suelen aparecer dos
"jorobas", y un buen umbral cae entre las dos. Una cifra útil para
describirlo es **qué proporción de huecos supera cada umbral**. Con los
siete huecos del visitante de la demo:

In [ ]:
huecos_ejemplo = pd.Series([5, 7, 38, 12, 8, 170, 3])   # minutos entre eventos del visitante de la demo
print(f"Mediana: {huecos_ejemplo.median():.1f} min")
# (huecos_ejemplo > 30) es True/False en cada hueco; su media es la proporción de True.
print(f"Proporción de huecos de más de 30 min: {(huecos_ejemplo > 30).mean():.1%}")

Dos de los siete huecos (38 y 170 minutos) superan los 30 minutos: el
28,6 %. Son los dos cortes de sesión de la demo. En el ejercicio 3 harás la
misma cuenta con 1,3 millones de huecos.

### Ejercicio 2 · Escribir la función de sesionización · completar código + interpretar

La demo tiene los tres pasos, uno por celda: copia su estructura dentro de
la función. Pistas:

```python
# pd.Timedelta(minutes=umbral_min) es una duración que se puede comparar con los huecos.
# serie.nunique() cuenta valores distintos (aquí, sesiones).
```

(a) Completa las tres líneas de la función `sesionizar`.

In [ ]:
# TU CÓDIGO
def sesionizar(eventos, umbral_min):
    """Asigna un número de sesión: empieza una nueva tras más de `umbral_min` sin actividad."""
    # 1. Ordenar (copy: trabajamos sobre una copia y no tocamos la tabla original).
    eventos = COMPLETA  # TODO: ordena por visitante y tiempo, y copia
    # 2. Hueco con el evento anterior del mismo visitante (nulo en su primer evento).
    hueco = COMPLETA  # TODO: hueco con el evento anterior del mismo visitante
    # 3. Inicio de sesión: hueco nulo (isna) o mayor que el umbral; | significa "o".
    nueva = COMPLETA  # TODO: True si empieza sesión
    # cumsum suma los True acumulados: sube en uno en cada inicio de sesión.
    eventos["session_id"] = nueva.cumsum()
    return eventos

(b) Probamos la función con un visitante que tiene eventos a las 9:00,
9:20, 10:05 y 10:10. Calcula antes a mano cuántas sesiones esperas con 30
y con 60 minutos.

In [ ]:
log_prueba = pd.DataFrame({
    "visitor_id": 7,
    "timestamp": pd.to_datetime(["2015-06-01 09:00", "2015-06-01 09:20",
                                 "2015-06-01 10:05", "2015-06-01 10:10"]),
    "event": ["view", "view", "view", "view"],
})
for umbral in [30, 60]:
    print(f"Umbral {umbral} min -> {sesionizar(log_prueba, umbral)['session_id'].nunique()} sesiones")

In [ ]:
# Comprobación
assert sesionizar(log_prueba, 30)["session_id"].nunique() == 2, \
    "Con 30 minutos deben salir 2 sesiones: los huecos son de 20, 45 y 5 minutos."
assert sesionizar(log_prueba, 60)["session_id"].nunique() == 1, \
    "Con 60 minutos debe salir 1 sesión: ningún hueco supera los 60 minutos."
print("✔ Correcto")

(c) Completa la sesionización del log real con 30 minutos y el número de
sesiones.

In [ ]:
# TU CÓDIGO
eventos = COMPLETA  # TODO: sesioniza el log con un umbral de 30 minutos
n_sesiones = COMPLETA  # TODO: número de sesiones distintas
print(f"Sesiones: {n_sesiones:,}")
eventos[["visitor_id", "timestamp", "event", "item_id", "session_id"]].head(8)

In [ ]:
# Comprobación
assert n_sesiones == 1_761_675, \
    "Deben salir 1 761 675 sesiones. Revisa que ordenas por visitante Y por tiempo."
print("✔ Correcto")

Pregunta: ¿qué saldría mal si te olvidaras de ordenar? Explícalo con dos
eventos de un mismo visitante.

**Tu respuesta:**

### Ejercicio 3 · ¿Cuánto importa el umbral? · completar código + interpretar

Si no has terminado el ejercicio 2, activa la celda de rescate: define
`sesionizar` y crea `eventos` sin duplicados y con las sesiones de 30
minutos.

In [ ]:
# Celda de rescate: cambia False por True solo si no has llegado hasta aquí.
USAR_RESCATE = False
if USAR_RESCATE:
    def sesionizar(eventos, umbral_min):
        """Asigna un número de sesión: empieza una nueva tras más de `umbral_min` sin actividad."""
        eventos = eventos.sort_values(["visitor_id", "timestamp"]).copy()
        hueco = eventos.groupby("visitor_id")["timestamp"].diff()
        nueva = hueco.isna() | (hueco > pd.Timedelta(minutes=umbral_min))
        eventos["session_id"] = nueva.cumsum()
        return eventos

    eventos = sesionizar(pd.read_parquet("events.parquet").drop_duplicates(), 30)
    print(f"Rescate cargado: {len(eventos):,} eventos y {eventos['session_id'].nunique():,} sesiones")

Pistas:

```python
# serie_booleana.groupby(ses["session_id"]).any(): True en cada sesión con al menos un True.
# serie_booleana.mean() es la proporción de True.
# hueco.dt.total_seconds() / 60 pasa una duración a minutos; .dropna() quita los nulos.
```

(a) El bucle sesioniza el log con cinco umbrales (tarda unos segundos).
Completa las tres cifras de cada fila: número de sesiones, eventos por
sesión y proporción de sesiones con alguna compra (entre todas las
sesiones).

In [ ]:
# TU CÓDIGO
filas = []
for umbral in [5, 10, 30, 60, 120]:
    ses = sesionizar(eventos[["visitor_id", "timestamp", "event"]], umbral)
    # True para cada sesión con al menos una compra.
    compra = ses["event"].eq("transaction").groupby(ses["session_id"]).any()
    filas.append({
        "umbral_min": umbral,
        "sesiones": COMPLETA,  # TODO: número de sesiones
        "eventos_por_sesion": COMPLETA,  # TODO: eventos entre sesiones
        "sesiones_con_compra": COMPLETA,  # TODO: proporción de sesiones con alguna compra
    })
sensibilidad = pd.DataFrame(filas).set_index("umbral_min")
sensibilidad

In [ ]:
# Comprobación
assert sensibilidad.loc[30, "sesiones"] == 1_761_675, "Con 30 minutos deben salir 1 761 675 sesiones."
assert round(sensibilidad.loc[30, "sesiones_con_compra"], 4) == 0.0081, \
    "Revisa sesiones_con_compra: es la media de compra (una fila por sesión), no de los eventos."
assert round(sensibilidad.loc[5, "sesiones_con_compra"], 4) == 0.0088, \
    "Con 5 minutos la proporción de sesiones con compra es 0,0088: ¿agrupas por sesión?"
print("✔ Correcto")

(b) Completa los huecos entre eventos consecutivos de un mismo visitante en
minutos (`hueco_min`) y la proporción que supera 5, 30 y 120 minutos, como
en el ejemplo de B2. `eventos` ya está ordenado por visitante y tiempo.

In [ ]:
# TU CÓDIGO
hueco = COMPLETA  # TODO: hueco con el evento anterior del mismo visitante
hueco_min = COMPLETA  # TODO: en minutos y sin los nulos
print(f"Huecos entre eventos consecutivos de un mismo visitante: {len(hueco_min):,}")
print(f"Mediana: {hueco_min.median():.1f} min")
for umbral in [5, 30, 120]:
    proporcion = COMPLETA  # TODO: proporción de huecos mayores que el umbral
    print(f"  más de {umbral:>3} min: {proporcion:.1%}")

In [ ]:
# Comprobación
assert len(hueco_min) == 1_348_061, "Deben salir 1 348 061 huecos: ¿has quitado los nulos con dropna()?"
assert round(hueco_min.median(), 1) == 2.3, "La mediana de los huecos es 2,3 minutos: ¿están en minutos?"
print("✔ Correcto")

Como los huecos van de segundos a meses, los dibujamos con escala
logarítmica en el eje horizontal. Las líneas son los cinco umbrales de (a).

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(hueco_min.clip(lower=0.01), bins=np.logspace(-2, 5.5, 61), color=AZUL)
for umbral in [5, 10, 30, 60, 120]:
    ax.axvline(umbral, color=NARANJA if umbral == 30 else GRIS, linestyle="--", linewidth=1.2)
ax.text(30, ax.get_ylim()[1] * 0.93, " 30 min", color=NARANJA, fontsize=9)
ax.set_xscale("log")
ax.set_xticks([0.1, 1, 10, 60, 24 * 60, 7 * 24 * 60, 30 * 24 * 60],
              ["6 s", "1 min", "10 min", "1 h", "1 día", "1 sem.", "1 mes"])
ax.set(title="Tiempo entre eventos consecutivos del mismo visitante",
       xlabel="hueco (escala log); líneas: umbrales de 5, 10, 30, 60 y 120 min",
       ylabel="número de huecos")
fig.tight_layout()
plt.show()

La barra aislada de la izquierda agrupa los huecos de menos de un segundo,
incluidos los de cero exactos: eventos registrados en el mismo milisegundo,
la mayoría compras de varios productos de un mismo pedido.

Preguntas: (i) ¿cuánto cambian el número de sesiones y la proporción de
sesiones con compra entre 5 y 120 minutos? ¿Cómo citarías la conversión por
sesión en un informe? (ii) Con el histograma, ¿por qué 30 minutos es una
elección razonable?

**Tu respuesta:**

### B3. Funnels

Si te has atascado antes, activa la celda de rescate: define `sesionizar` y
crea `eventos` sin duplicados y con las sesiones de 30 minutos.

In [ ]:
# Celda de rescate: cambia False por True solo si no has llegado hasta aquí.
USAR_RESCATE = False
if USAR_RESCATE:
    def sesionizar(eventos, umbral_min):
        """Asigna un número de sesión: empieza una nueva tras más de `umbral_min` sin actividad."""
        eventos = eventos.sort_values(["visitor_id", "timestamp"]).copy()
        hueco = eventos.groupby("visitor_id")["timestamp"].diff()
        nueva = hueco.isna() | (hueco > pd.Timedelta(minutes=umbral_min))
        eventos["session_id"] = nueva.cumsum()
        return eventos

    eventos = sesionizar(pd.read_parquet("events.parquet").drop_duplicates(), 30)
    print(f"Rescate cargado: {len(eventos):,} eventos y {eventos['session_id'].nunique():,} sesiones")

Un **funnel** (embudo) describe un recorrido como una serie de etapas por
las que hay que pasar para llegar a un objetivo. En comercio electrónico, el
más básico es:

**vista de producto → añadir al carrito → compra**

Para cada etapa se calculan tres cifras:

- la **conversión de etapa**: qué proporción de los que llegaron a la etapa
  anterior llega a esta;
- el ***drop-off*** (abandono en la etapa): 1 menos la conversión de etapa;
- la **conversión total**: qué proporción de los que empezaron llega a esta
  etapa.

Si $n_k$ es el número de sesiones (o visitantes) que alcanzan la etapa $k$:

$$
\text{conversión}_k = \frac{n_k}{n_{k-1}}, \qquad
\text{drop-off}_k = 1 - \frac{n_k}{n_{k-1}}, \qquad
\text{conversión total}_k = \frac{n_k}{n_1}.
$$

**Ejemplo resuelto: diez sesiones.** Todas ven algún producto; cuatro
añaden algo al carrito (S1, S3, S6 y S8); dos compran (S1 y S6). A mano: de
10 a 4 es una conversión del 40 % y un *drop-off* del 60 %; de 4 a 2, una
conversión del 50 %; en total, 2 de 10, un 20 %. Cada fila de la tabla es
una sesión y vale 1 si llega a la etapa:

In [ ]:
sesiones_ejemplo = pd.DataFrame({
    "view":        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    "addtocart":   [1, 0, 1, 0, 0, 1, 0, 1, 0, 0],
    "transaction": [1, 0, 0, 0, 0, 1, 0, 0, 0, 0],
}, index=[f"S{i}" for i in range(1, 11)])
llegan = sesiones_ejemplo.sum()   # sesiones que llegan a cada etapa: 10, 4, 2
funnel_ej = pd.DataFrame({
    "sesiones": llegan,
    # shift(1) desplaza una posición: cada etapa se divide por la anterior.
    "conversion_etapa": llegan / llegan.shift(1),
    # iloc[0] es la primera etapa (las 10 sesiones con vista).
    "conversion_total": llegan / llegan.iloc[0],
})
funnel_ej["drop_off"] = 1 - funnel_ej["conversion_etapa"]
funnel_ej

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
fig, ax = plt.subplots(figsize=(8, 3))
nombres = {"view": "vista", "addtocart": "carrito", "transaction": "compra"}
for fila, etapa in enumerate(sesiones_ejemplo.columns):
    y = 2 - fila
    for x, (sesion, llega) in enumerate(sesiones_ejemplo[etapa].items()):
        ax.scatter(x, y, s=330, color=ESTILO[etapa][0] if llega else "white",
                   edgecolors=ESTILO[etapa][0], linewidths=1.5, zorder=3)
        if fila == 0:
            ax.text(x, 2.45, sesion, ha="center", fontsize=9)
    ax.text(-0.8, y, f"{nombres[etapa]}: {sesiones_ejemplo[etapa].sum()}", ha="right",
            va="center", fontsize=10)
for y, etapa in zip([1.5, 0.5], ["addtocart", "transaction"]):
    fila = funnel_ej.loc[etapa]
    ax.text(9.8, y, f"conversión {100 * fila['conversion_etapa']:.0f} %\n"
            f"drop-off {100 * fila['drop_off']:.0f} %",
            va="center", fontsize=9, color=GRIS)
ax.set(xlim=(-3.5, 11.8), ylim=(-0.5, 2.8))
ax.axis("off")
ax.set_title("Diez sesiones: rellenas las que llegan a cada etapa", fontsize=10)
fig.tight_layout()
plt.show()

El embudo se ve al leer de arriba abajo: cada fila tiene menos círculos
rellenos que la anterior. La conversión de etapa compara una fila con la
anterior; la conversión total, con la primera. Y la conversión total es el
**producto** de las conversiones de etapa: 0,4 × 0,5 = 0,2. Por eso un
funnel se lee etapa a etapa: una conversión total baja puede deberse a una
sola etapa muy mala o a varias mediocres, y la mejora que conviene hacer es
distinta en cada caso.

Para no repetir estas cuentas, la función `tabla_funnel` recibe cuántos
llegan a cada etapa y devuelve la tabla completa. La probamos con las diez
sesiones:

In [ ]:
def tabla_funnel(llegan):
    """Conversión de etapa, drop-off y conversión total a partir de los que llegan a cada etapa."""
    tabla = pd.DataFrame({"llegan": llegan})
    tabla["conversion_etapa"] = tabla["llegan"] / tabla["llegan"].shift(1)
    tabla["drop_off"] = 1 - tabla["conversion_etapa"]
    tabla["conversion_total"] = tabla["llegan"] / tabla["llegan"].iloc[0]
    return tabla


tabla_funnel(sesiones_ejemplo.sum())

**Del log al funnel.** Un log real no trae esa tabla de unos y ceros: trae
una fila por evento. Para saber si una sesión tiene **al menos** un evento
de cada tipo, `pd.crosstab` cuenta los eventos de cada sesión y tipo, y
`.gt(0)` ("mayor que 0") convierte cada recuento en `True`/`False`. Además,
el funnel es **estricto**: una sesión llega a una etapa solo si también ha
pasado por las anteriores, y eso se escribe con `&` ("y"). Con las diez
sesiones escritas como log (16 eventos):

In [ ]:
log_funnel_ejemplo = pd.DataFrame({
    "session_id": list(range(1, 11)) + [1, 3, 6, 8] + [1, 6],
    "event": ["view"] * 10 + ["addtocart"] * 4 + ["transaction"] * 2,
})
por_sesion_ejemplo = pd.crosstab(log_funnel_ejemplo["session_id"],
                                 log_funnel_ejemplo["event"]).gt(0)
print(por_sesion_ejemplo.head(3))
llegan_ejemplo = pd.Series({
    "view": por_sesion_ejemplo["view"].sum(),   # sum() cuenta los True
    "addtocart": (por_sesion_ejemplo["view"] & por_sesion_ejemplo["addtocart"]).sum(),
    "transaction": (por_sesion_ejemplo["view"] & por_sesion_ejemplo["addtocart"]
                    & por_sesion_ejemplo["transaction"]).sum(),
})
tabla_funnel(llegan_ejemplo)

La misma tabla que antes: 10, 4 y 2.

**Funnel por sesión frente a funnel por visitante.** La misma pregunta, "¿qué
proporción llega a comprar?", tiene dos respuestas según la **unidad**. Dos
visitantes y cuatro sesiones: el visitante 1 entra tres veces y compra en la
tercera; el visitante 2 entra una vez y solo mira.

In [ ]:
mini = pd.DataFrame({
    "visitante": ["visitante 1"] * 3 + ["visitante 2"],
    "sesion": ["sesión 1", "sesión 2", "sesión 3", "sesión 4"],
    "recorrido": ["VV", "VC", "VCT", "V"],
})
mini["compra"] = mini["recorrido"].str.contains("T")   # True si el recorrido tiene una T
print(f"Conversión por sesión:    {mini['compra'].sum()} de {len(mini)} sesiones"
      f" = {mini['compra'].mean():.0%}")
# any(): True si al menos una sesión del visitante tiene compra.
compra_vis = mini.groupby("visitante")["compra"].any()
print(f"Conversión por visitante: {compra_vis.sum()} de {len(compra_vis)} visitantes"
      f" = {compra_vis.mean():.0%}")

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
fig, ax = plt.subplots(figsize=(8, 2.8))
letras = {"V": "view", "C": "addtocart", "T": "transaction"}
for i, fila in mini.iterrows():
    x0 = i * 2.2 + (0.6 if fila["visitante"] == "visitante 2" else 0)
    ax.add_patch(plt.Rectangle((x0, 0), 1.9, 1, facecolor="#e8f3ee" if fila["compra"]
                               else "#f4f3f0", edgecolor=GRIS, linewidth=1))
    ax.text(x0 + 0.95, 1.12, fila["sesion"], ha="center", fontsize=9)
    for j, letra in enumerate(fila["recorrido"]):
        color, marca, _ = ESTILO[letras[letra]]
        xc = x0 + 0.95 + (j - (len(fila["recorrido"]) - 1) / 2) * 0.5
        ax.scatter(xc, 0.5, s=150, color=color, marker=marca, edgecolors="white", zorder=3)
        ax.text(xc, 0.18, letra, ha="center", fontsize=9, fontweight="bold", color=color)
for nombre, (xa, xb) in {"visitante 1": (0, 6.3), "visitante 2": (7.2, 9.1)}.items():
    ax.plot([xa, xb], [-0.25, -0.25], color=GRIS, linewidth=1.5)
    ax.text((xa + xb) / 2, -0.55, f"{nombre}: {'compra' if compra_vis[nombre] else 'no compra'}",
            ha="center", fontsize=10)
ax.set(xlim=(-0.2, 9.3), ylim=(-0.8, 1.4))
ax.axis("off")
fig.tight_layout()
plt.show()

Una sesión de cuatro compra (25 %), pero un visitante de dos compra (50 %).
Las sesiones 1 y 2 cuentan como fracasos en el funnel por sesión, aunque
formaban parte del recorrido de alguien que acabó comprando. Las dos cifras
son correctas, pero responden a preguntas distintas: la conversión por
sesión dice cómo de eficaz es **cada visita**; la conversión por visitante,
qué proporción de **personas** (aquí, de cookies) acaba comprando. Regla:
**el numerador y el denominador tienen la misma unidad**. Dividir
compradores entre sesiones da un número que no significa nada.

**Tiempo entre etapas.** ¿Cuánto tardan en decidirse los que compran? Para
cada sesión se busca el momento de la **primera** vista, el primer carrito y
la primera compra: `groupby(["session_id", "event"])["timestamp"].min()` da
una fila por sesión y tipo de evento, y `unstack()` pone cada tipo de evento
en una columna. Con una sesión de cuatro eventos:

In [ ]:
una_sesion_ejemplo = pd.DataFrame({
    "session_id": 1,
    "timestamp": pd.to_datetime(["2015-06-01 10:00", "2015-06-01 10:02",
                                 "2015-06-01 10:05", "2015-06-01 10:09"]),
    "event": ["view", "view", "addtocart", "transaction"],
})
primeras_ejemplo = una_sesion_ejemplo.groupby(["session_id", "event"])["timestamp"].min().unstack()
print(primeras_ejemplo)
minutos_hasta_carrito = (primeras_ejemplo["addtocart"] - primeras_ejemplo["view"]).dt.total_seconds() / 60
minutos_hasta_compra = (primeras_ejemplo["transaction"] - primeras_ejemplo["addtocart"]).dt.total_seconds() / 60
print(f"Vista -> carrito: {minutos_hasta_carrito.iloc[0]:.0f} min;  "
      f"carrito -> compra: {minutos_hasta_compra.iloc[0]:.0f} min")

Con miles de sesiones, los tiempos serán muy asimétricos, como el gasto de
la clase 1: unas pocas sesiones tardan horas y suben mucho la media. Por eso
los tiempos se describen con la **mediana** y los percentiles. Si un paso
tiene una mediana alta o una cola muy larga, es una señal de **fricción**
(por ejemplo, un proceso de pago lento).

### Ejercicio 4 · El funnel por sesión · completar código + interpretar

El ejemplo de las diez sesiones escritas como log tiene el mismo `crosstab`
y la misma serie `llegan`. La celda tarda unos segundos: `eventos` tiene 1,76
millones de sesiones.

(a) Completa `por_sesion` (una fila por sesión, una columna por tipo de
evento, `True` si hay al menos uno) y las sesiones que llegan al carrito y a
la compra en el funnel estricto.

In [ ]:
# TU CÓDIGO
por_sesion = COMPLETA  # TODO: True si la sesión tiene al menos un evento de cada tipo
llegan_sesion = pd.Series({
    "view": por_sesion["view"].sum(),
    "addtocart": COMPLETA,  # TODO: sesiones con vista y carrito
    "transaction": COMPLETA,  # TODO: sesiones con las tres etapas
})
tabla_funnel(llegan_sesion)

In [ ]:
# Comprobación
assert len(por_sesion) == 1_761_675, "por_sesion debe tener una fila por sesión (1 761 675)."
assert list(llegan_sesion) == [1_755_781, 38_726, 11_143], \
    "Revisa llegan_sesion: cada etapa exige también las anteriores (&)."
print("✔ Correcto")

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
fig, axes = plt.subplots(1, 2, figsize=(10, 3), gridspec_kw={"width_ratios": [3, 2]})
valores = llegan_sesion / llegan_sesion.iloc[0] * 100
axes[0].barh(["1. vista", "2. carrito", "3. compra"][::-1], valores.to_numpy()[::-1],
             color=[AQUA, NARANJA, AZUL], height=0.6)
for i, v in enumerate(valores.to_numpy()[::-1]):
    axes[0].text(v + 1, i, f"{v:.2f} %".replace(".", ","), va="center", color="#0b0b0b")
axes[0].set(title="Conversión total (% de sesiones con vista)", xlabel="% de sesiones",
            xlim=(0, 118))
etapa = 100 * llegan_sesion.iloc[1:].to_numpy() / llegan_sesion.iloc[:-1].to_numpy()
axes[1].barh(["vista → carrito", "carrito → compra"][::-1], etapa[::-1],
             color=[AQUA, NARANJA], height=0.6)
for i, v in enumerate(etapa[::-1]):
    axes[1].text(v + 1, i, f"{v:.1f} %".replace(".", ","), va="center", color="#0b0b0b")
axes[1].set(title="Conversión de cada etapa", xlabel="% de la etapa anterior", xlim=(0, 40))
for ax in axes:
    ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()

Los dos paneles muestran el mismo funnel. El de la izquierda, el habitual en
los informes, mide todo respecto a las sesiones con vista: las barras del
carrito y la compra casi no se ven. El de la derecha mide cada etapa
respecto a la anterior y separa las dos pérdidas.

Preguntas: (i) ¿en qué etapa se pierden más sesiones en volumen y cuál crees
que es más accionable? Cita las cifras. (ii) Un funnel tiene conversiones de
etapa del 10 %, 50 % y 40 %. ¿Cuál es la conversión total? Si cada mejora
posible sube su etapa 5 puntos porcentuales (por ejemplo, del 10 % al 15 %),
¿cuál aumenta más el total?

**Tu respuesta:**

### Ejercicio 5 · Lo que el funnel estricto deja fuera y el funnel por visitante · completar código + interpretar

Si no has terminado el ejercicio 4, activa la celda de rescate: crea
`por_sesion` y `llegan_sesion`.

In [ ]:
# Celda de rescate: cambia False por True solo si no has llegado hasta aquí.
USAR_RESCATE = False
if USAR_RESCATE:
    por_sesion = pd.crosstab(eventos["session_id"], eventos["event"]).gt(0)
    llegan_sesion = pd.Series({
        "view": por_sesion["view"].sum(),
        "addtocart": (por_sesion["view"] & por_sesion["addtocart"]).sum(),
        "transaction": (por_sesion["view"] & por_sesion["addtocart"]
                        & por_sesion["transaction"]).sum(),
    })
    print("Rescate cargado: llegan_sesion =", list(llegan_sesion))

Al exigir que una sesión pase por todas las etapas, algunas sesiones quedan
fuera del funnel. Ya se ve en la primera etapa: llegan 1 755 781 sesiones y
no las 1 761 675 que hay, porque 5 894 no tienen ninguna vista (empiezan en
el carrito o en la compra). Y también quedan fuera **compras**: sesiones que
compran sin carrito o sin vista en la misma sesión. Por eso hay dos formas de
medir la "conversión por sesión", y al citar una hay que decir cuál:

- **sesiones con alguna compra / todas las sesiones**: cuenta cualquier
  compra;
- **funnel estricto / sesiones con vista**: solo las que pasan por vista,
  carrito y compra.

Pista:

```python
# ~mascara invierte True/False: compra & ~por_sesion["addtocart"] son las sesiones con compra y sin carrito.
```

(a) Completa las sesiones con compra sin carrito y sin vista, y las dos
formas de medir la conversión por sesión: sesiones con alguna compra entre
todas las sesiones, y sesiones del funnel estricto entre las sesiones con
vista.

In [ ]:
# TU CÓDIGO
compra = por_sesion["transaction"]
sin_carrito = COMPLETA  # TODO: sesiones con compra y sin carrito
sin_vista = COMPLETA  # TODO: sesiones con compra y sin vista
conv_alguna = COMPLETA  # TODO: sesiones con alguna compra / todas las sesiones
conv_estricta = COMPLETA  # TODO: funnel estricto / sesiones con vista
print(f"Sesiones con compra: {compra.sum():,}")
print(f"  sin añadir al carrito en la sesión: {sin_carrito:,}")
print(f"  sin ninguna vista en la sesión:     {sin_vista:,}")
print(f"  que el funnel estricto deja fuera:  {compra.sum() - llegan_sesion['transaction']:,}")
print(f"Sesiones con alguna compra / todas las sesiones: {conv_alguna:.2%}")
print(f"Funnel estricto / sesiones con vista:           {conv_estricta:.2%}")

In [ ]:
# Comprobación
assert (sin_carrito, sin_vista) == (2365, 1485), "Revisa las máscaras: compra & ~por_sesion[...]."
assert round(conv_alguna, 4) == 0.0081 and round(conv_estricta, 4) == 0.0063, \
    "Revisa las conversiones: alguna compra entre todas las sesiones; estricto entre sesiones con vista."
print("✔ Correcto")

(b) Completa el funnel **por visitante**: lo mismo que en el ejercicio 4,
pero con una fila por visitante (`visitor_id`) en lugar de por sesión. La
celda tarda unos segundos.

In [ ]:
# TU CÓDIGO
por_visitante_ev = COMPLETA  # TODO: una fila por visitante
llegan_visitante = pd.Series({
    "view": por_visitante_ev["view"].sum(),
    "addtocart": COMPLETA,  # TODO: visitantes con vista y carrito
    "transaction": COMPLETA,  # TODO: con las tres etapas
})
tabla_funnel(llegan_visitante)

In [ ]:
# Comprobación
assert list(llegan_visitante) == [1_404_179, 34_401, 10_228], \
    "Revisa llegan_visitante: es el funnel estricto con una fila por visitante."
print("✔ Correcto")

Las dos tablas, lado a lado, y cuántos visitantes completan el funnel por
visitante **sin que ninguna de sus sesiones lo complete** (sus etapas están
repartidas entre visitas). Cómo lo cuenta la celda: (1) marca los visitantes
que completan el funnel por visitante; (2) busca el visitante de cada sesión
que completa el funnel estricto (`first()` de cada sesión); (3) se queda con
los del paso 1 que **no** están en esa lista (`isin` dice si un valor está en
una lista y `~` lo invierte).

In [ ]:
comparacion = pd.DataFrame({
    "por_sesion": tabla_funnel(llegan_sesion)["conversion_etapa"],
    "por_visitante": tabla_funnel(llegan_visitante)["conversion_etapa"],
})
comparacion.loc["total"] = [llegan_sesion.iloc[-1] / llegan_sesion.iloc[0],
                            llegan_visitante.iloc[-1] / llegan_visitante.iloc[0]]
print(comparacion.dropna())   # conversión de cada etapa y total

# Visitantes que completan el funnel por visitante...
completo_visitante = (por_visitante_ev["view"] & por_visitante_ev["addtocart"]
                      & por_visitante_ev["transaction"])
# ...y visitantes con al menos una sesión que completa el funnel estricto.
sesion_completa = por_sesion["view"] & por_sesion["addtocart"] & por_sesion["transaction"]
visitante_de_sesion = eventos.groupby("session_id")["visitor_id"].first()
con_sesion_completa = visitante_de_sesion[sesion_completa].unique()
# isin: True si el visitante está en la lista; ~ lo invierte.
repartido = completo_visitante & ~completo_visitante.index.isin(con_sesion_completa)
print(f"\nVisitantes que completan el funnel por visitante: {completo_visitante.sum():,}")
print(f"  sin ninguna sesión que lo complete (etapas repartidas): {repartido.sum():,}")

Preguntas: (i) ¿cómo puede alguien comprar sin añadir al carrito en la misma
sesión? Da dos explicaciones. (ii) ¿Por qué la conversión por visitante
(0,73 %) es mayor que por sesión (0,63 %)? (iii) Un informe divide los
10 228 compradores entre las 1 755 781 sesiones con vista. ¿Qué le dices?

**Tu respuesta:**

### Ejercicio 6 · Tiempo entre etapas · completar código + interpretar

Pistas:

```python
# tabla.dropna(subset=["a", "b"]) quita las filas con algún nulo en esas columnas.
# tabla[(tabla["a"] <= tabla["b"]) & (tabla["b"] <= tabla["c"])] exige las dos condiciones a la vez.
```

(a) Completa `primeras` (momento del primer evento de cada tipo en cada
sesión), las sesiones que tienen las tres etapas y, de ellas, las que las
tienen en orden (primera vista ≤ primer carrito ≤ primera compra).

In [ ]:
# TU CÓDIGO
primeras = COMPLETA  # TODO: primer evento de cada tipo, tipos en columnas
completas = COMPLETA  # TODO: sesiones con las tres etapas
completas = COMPLETA  # TODO: en orden
print(f"Sesiones con las tres etapas en orden: {len(completas):,}")

In [ ]:
# Comprobación
assert len(completas) == 10_411, \
    "Deben quedar 10 411 sesiones: con las tres etapas (sin nulos) y en orden temporal."
print("✔ Correcto")

Son menos que las 11 143 sesiones compradoras del funnel del ejercicio 4
porque aquí exigimos además que la primera vista, el primer carrito y la
primera compra ocurran **en ese orden**; el funnel solo comprobaba que la
sesión tuviera las tres etapas.

Minutos de cada paso, su resumen y su distribución (escala logarítmica). En
la figura, la línea continua es la mediana y la punteada, la media.

In [ ]:
minutos = pd.DataFrame({
    "vista -> carrito": (completas["addtocart"] - completas["view"]).dt.total_seconds() / 60,
    "carrito -> compra": (completas["transaction"] - completas["addtocart"]).dt.total_seconds() / 60,
})
resumen_min = minutos.describe(percentiles=[0.25, 0.5, 0.75, 0.9]).T   # .T: filas <-> columnas
resumen_min["count"] = resumen_min["count"].astype(int)   # el recuento, como número entero
print(resumen_min.to_string(float_format="{:,.1f}".format))

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for ax, (paso, valores), color in zip(axes, minutos.items(), [AZUL, NARANJA]):
    ax.hist(valores.clip(lower=0.1), bins=np.logspace(-1, 3, 33), color=color)
    ax.axvline(valores.median(), color="#0b0b0b", linewidth=1.5)
    ax.axvline(valores.mean(), color="#0b0b0b", linewidth=1.5, linestyle=":")
    ax.text(valores.median(), ax.get_ylim()[1] * 0.92, f" mediana {valores.median():.1f}".replace(".", ","),
            fontsize=9)
    ax.text(valores.mean(), ax.get_ylim()[1] * 0.80, f" media {valores.mean():.1f}".replace(".", ","), fontsize=9)
    ax.set(xscale="log", title=paso, xlabel="minutos (escala log)")
    ax.set_xticks([0.1, 1, 10, 100, 1000], ["≤ 0,1", "1", "10", "100", "1000"])
axes[0].set_ylabel("sesiones")
fig.tight_layout()
plt.show()

Preguntas: (i) ¿cuánto tarda la sesión típica en pasar de la vista al
carrito y del carrito a la compra? ¿Usarías la media o la mediana en un
informe y por qué? (ii) ¿Ves un cuello de botella en el pago?

**Tu respuesta:**

### B4. Cohortes y retención

Si te has atascado antes, activa la celda de rescate: crea `eventos` sin
duplicados y con las sesiones de 30 minutos.

In [ ]:
# Celda de rescate: cambia False por True solo si no has llegado hasta aquí.
USAR_RESCATE = False
if USAR_RESCATE:
    def sesionizar(eventos, umbral_min):
        """Asigna un número de sesión: empieza una nueva tras más de `umbral_min` sin actividad."""
        eventos = eventos.sort_values(["visitor_id", "timestamp"]).copy()
        hueco = eventos.groupby("visitor_id")["timestamp"].diff()
        nueva = hueco.isna() | (hueco > pd.Timedelta(minutes=umbral_min))
        eventos["session_id"] = nueva.cumsum()
        return eventos

    eventos = sesionizar(pd.read_parquet("events.parquet").drop_duplicates(), 30)
    print(f"Rescate cargado: {len(eventos):,} eventos y {eventos['session_id'].nunique():,} sesiones")

El funnel mira lo que pasa **dentro** de una visita. La **retención** mira lo
que pasa **entre** visitas: ¿vuelven los visitantes? Para medirla se usan
**cohortes**: grupos de clientes que empezaron en el mismo periodo. Una
**cohorte de adquisición** agrupa a los clientes por la fecha de su primera
interacción (primera visita, primera compra, fecha de alta).

¿Por qué agrupar por fecha de inicio? Porque los clientes que llegaron en
enero y los que llegaron en junio han tenido tiempos de observación
distintos. Compararlos mezclados confunde; compararlos por cohorte, **a la
misma edad**, permite ver si el negocio retiene mejor o peor con el tiempo.

**Ejemplo resuelto: una tabla de cohortes con números redondos.** Cada fila
es una cohorte (los clientes que llegaron ese mes) y cada columna, los meses
transcurridos desde su llegada. La primera tabla da clientes activos; la
segunda divide cada fila por su mes 0: es la **retención**.

In [ ]:
cohortes_ej = pd.DataFrame(
    [[1000, 400, 300, 250], [800, 280, 200, np.nan],
     [1200, 360, np.nan, np.nan], [900, np.nan, np.nan, np.nan]],
    index=["enero", "febrero", "marzo", "abril"],
    columns=["mes 0", "mes 1", "mes 2", "mes 3"])
# Recuentos de clientes sin decimales (los huecos, NaN, son meses aún no observados).
print(cohortes_ej.to_string(float_format="{:.0f}".format))
print()
cohortes_ej.div(cohortes_ej["mes 0"], axis=0)   # cada fila entre su mes 0

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
ret_ej = cohortes_ej.div(cohortes_ej["mes 0"], axis=0)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))
titulos = ["Por filas: la curva de una cohorte", "Por columnas: cohortes a la misma edad",
           "Por diagonales: un mes del calendario"]
for n, (ax, titulo) in enumerate(zip(axes, titulos)):
    ax.imshow(ret_ej.to_numpy(dtype=float), cmap="Blues", vmin=0, vmax=1.1)
    for (i, j), v in np.ndenumerate(ret_ej.to_numpy(dtype=float)):
        if not np.isnan(v):
            ax.text(j, i, f"{100 * v:.0f} %", ha="center", va="center", fontsize=9,
                    color="white" if v > 0.6 else "#0b0b0b")
    if n == 0:     # fila de enero
        ax.add_patch(plt.Rectangle((-0.5, -0.5), 4, 1, fill=False, edgecolor=NARANJA, linewidth=3))
    elif n == 1:   # columna del mes 1
        ax.add_patch(plt.Rectangle((0.5, -0.5), 1, 3, fill=False, edgecolor=NARANJA, linewidth=3))
    else:          # diagonal de abril: enero mes 3, febrero mes 2, marzo mes 1, abril mes 0
        for i in range(4):
            ax.add_patch(plt.Rectangle((2.5 - i, -0.5 + i), 1, 1, fill=False,
                                       edgecolor=NARANJA, linewidth=3))
    ax.set_xticks(range(4), ret_ej.columns, fontsize=9)
    ax.set_yticks(range(4), ret_ej.index, fontsize=9)
    ax.set_title(titulo, fontsize=10)
    ax.grid(False)
fig.tight_layout()
plt.show()

Una tabla de cohortes se lee en tres direcciones:

- **Por filas**: la curva de retención de una cohorte. De los clientes de
  enero, al mes siguiente vuelve el 40 %, a los dos meses el 30 % y a los
  tres el 25 %. La caída es fuerte al principio y se suaviza.
- **Por columnas**: cohortes a la misma edad. La retención del mes 1 baja de
  enero (40 %) a febrero (35 %) y marzo (30 %): algo ha cambiado (la calidad
  de los clientes captados, la experiencia, la competencia).
- **Por diagonales**: cada diagonal es un mes del calendario. La recuadrada
  es abril. Si una diagonal entera fuera peor, apuntaría a un problema de
  ese mes (una caída de la web, una campaña de un competidor).

La forma triangular no es un defecto: la cohorte de abril solo lleva un mes
y **no se puede saber aún** su retención a tres meses.

**Construir la tabla desde un log.** Usaremos cohortes **semanales**: cada
visitante pertenece a la cohorte de la semana (de lunes a domingo) de su
primera visita, y está activo en la semana $k$ si tiene algún evento $k$
semanas después. Hacen falta tres columnas: la **semana** de cada evento
(su lunes), la **cohorte** del visitante (su primera semana, igual en todas
sus filas) y las **semanas desde el alta** (el "alta" de un visitante
anónimo es su primera visita). Un log mínimo de tres visitantes en mayo de
2015 (el 4 de mayo fue lunes):

In [ ]:
# Cinco eventos de tres visitantes (el lunes 4 de mayo de 2015 es el primer lunes).
mini_log = pd.DataFrame({
    "visitor_id": ["A", "A", "B", "B", "C"],
    "timestamp": pd.to_datetime(["2015-05-06", "2015-05-14", "2015-05-11",
                                 "2015-05-29", "2015-05-09"]),
})
# 1. Semana del evento: la fecha menos su día de la semana da el lunes.
#    weekday vale 0 el lunes, 1 el martes... y 6 el domingo.
dia_semana = mini_log["timestamp"].dt.weekday
mini_log["semana"] = mini_log["timestamp"] - pd.to_timedelta(dia_semana, unit="D")
# 2. Cohorte: la semana más temprana del visitante. transform("min") calcula el
#    mínimo de cada visitante y lo copia en todas sus filas.
mini_log["cohorte"] = mini_log.groupby("visitor_id")["semana"].transform("min")
# 3. Semanas desde el alta: días entre las dos fechas, divididos entre 7 sin decimales (//).
mini_log["semanas_desde_alta"] = (mini_log["semana"] - mini_log["cohorte"]).dt.days // 7
mini_log

Las tres operaciones nuevas:

- **El lunes de la semana.** `dt.weekday` da el día de la semana como un
  número (lunes 0… domingo 6). Restar a una fecha ese número de días lleva
  siempre al lunes: al miércoles 6 (weekday 2) le restamos 2 días y queda el
  lunes 4. Con un log real, las marcas temporales tienen hora; para que la
  semana empiece a las 00:00 del lunes se quita la hora antes con
  `dt.normalize()`.
- **`transform("min")` frente a `min()`.** `groupby("visitor_id")["semana"].min()`
  devolvería una fila por visitante (3 filas). `transform("min")` calcula el
  mismo mínimo pero lo **copia en cada fila** del visitante (5 filas), de
  modo que se puede guardar como una columna más del log.
- **`// 7`** es la división entera: 14 días // 7 = 2 semanas; 3 días // 7 = 0.

Después se cuentan, para cada cohorte y cada semana, los visitantes
**distintos** (`nunique`), se pasan las semanas a columnas (`unstack`) y se
divide cada fila por su semana 0:

In [ ]:
# Visitantes distintos de cada cohorte en cada semana desde el alta.
activos_mini = mini_log.groupby(["cohorte", "semanas_desde_alta"])["visitor_id"].nunique()
# unstack pasa las semanas a columnas; fill_value=0 si nadie vuelve esa semana.
activos_mini = activos_mini.unstack(fill_value=0)
print(activos_mini)
print()
# Retención: cada fila dividida por su semana 0 (axis=0: fila a fila).
activos_mini.div(activos_mini[0], axis=0)

La cohorte del 4 de mayo retiene el 50 % en la semana 1 (A vuelve, C no); la
del 11 de mayo, el 0 % en la semana 1 y el 100 % en la 2. Aquí `fill_value=0`
pone un 0 donde nadie vuelve, porque en este mini log todas las semanas
están observadas. **Con un log real no se usa**: las semanas que una cohorte
reciente aún no ha vivido no son un 0 % de retención, sino un dato que no
existe todavía, y deben quedar vacías (`NaN`), como el triángulo de la tabla
de cuatro meses. "Activo en la
semana $k$" **no es acumulativo**: B no vuelve en la semana 1 y sí en la 2,
así que una fila puede subir de una semana a otra.

**Censura al inicio de los datos.** El log empieza el 3 de mayo de 2015,
pero la tienda existía antes. Para muchos visitantes de las primeras semanas,
la "primera visita" que vemos no es la primera de verdad: ya eran
visitantes habituales. Es el mismo fenómeno que la antigüedad de la clase 1:
al principio del periodo observado no sabemos quién es realmente nuevo, y
las primeras cohortes parecen más fieles de lo que serían. Por la misma
razón, para resumir varias cohortes en una curva media hay que usar
**solo cohortes con las mismas semanas observadas**: si no, cada punto de la
curva se apoyaría en un grupo distinto de cohortes.

### Ejercicio 7 · Cohortes semanales y retención · completar código + interpretar

El mini log de arriba tiene las tres columnas y el recuento con tres
visitantes: es el mismo código, con dos cambios: `dt.normalize()` en la
primera línea y `unstack()` **sin** `fill_value=0`, para que las semanas aún
no observadas queden vacías.

(a) Completa las tres columnas de `eventos`: `semana`, `cohorte` y
`semanas_desde_alta`.

In [ ]:
# TU CÓDIGO
dia_semana = eventos["timestamp"].dt.weekday
eventos["semana"] = COMPLETA  # TODO: lunes de la semana (sin hora)
eventos["cohorte"] = COMPLETA  # TODO: primera semana del visitante, en todas sus filas
eventos["semanas_desde_alta"] = COMPLETA  # TODO: semanas entre la semana y la cohorte
eventos[["visitor_id", "timestamp", "semana", "cohorte", "semanas_desde_alta"]].head()

(b) La celda deja fuera la primera semana del log (solo tiene un domingo) y
la última (está incompleta): las cohortes completas van de la semana del 4
de mayo a la del 7 de septiembre. Completa `activos` (visitantes distintos
por cohorte y semana, con las semanas en columnas) y `retencion`.

In [ ]:
# TU CÓDIGO
en_cohortes = eventos[(eventos["cohorte"] >= "2015-05-04") & (eventos["semana"] <= "2015-09-07")]
activos = COMPLETA  # TODO: visitantes distintos por cohorte y semana
retencion = COMPLETA  # TODO: cada fila entre su semana 0
print(f"Cohortes: {len(retencion)}")

In [ ]:
# Comprobación
assert len(retencion) == 19, "Deben salir 19 cohortes. ¿Has quitado la hora con dt.normalize()?"
assert activos.iloc[0, 0] == 71_389, "La cohorte del 4 de mayo tiene 71 389 visitantes en la semana 0."
assert round(retencion.iloc[0, 1], 3) == 0.048, "Revisa retencion: cada fila dividida por su semana 0."
assert retencion[12].notna().sum() == 7, \
    "Solo 7 cohortes han llegado a la semana 12: usa unstack() sin fill_value=0 (las demás, vacías)."
print("✔ Correcto")

Las ocho primeras cohortes y sus ocho primeras semanas, en porcentaje:

In [ ]:
tabla_ret = (100 * retencion.iloc[:8, :8]).round(1)
tabla_ret.index = tabla_ret.index.strftime("%d %b")
tabla_ret.insert(0, "visitantes", activos.iloc[:8, 0].astype(int).to_numpy())
print(tabla_ret.to_string(float_format="{:.1f}".format))
print(f"\nSemana 1, en las 18 cohortes que ya la tienen: entre {retencion[1].min():.1%} "
      f"y {retencion[1].max():.1%}")

El mapa de calor con todas las cohortes y semanas (sin la semana 0, que
siempre vale 100 % y apagaría el resto de colores):

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
fig, ax = plt.subplots(figsize=(9, 5.2))
datos = 100 * retencion.loc[:, 1:].to_numpy(dtype=float)
im = ax.imshow(datos, cmap="Blues", vmin=0, vmax=5, aspect="auto")
for (i, j), v in np.ndenumerate(datos):
    if not np.isnan(v):
        ax.text(j, i, f"{v:.1f}".replace(".", ","), ha="center", va="center", fontsize=7,
                color="white" if v > 3 else "#0b0b0b")
ax.set_xticks(range(datos.shape[1]), range(1, datos.shape[1] + 1), fontsize=8)
ax.set_yticks(range(len(retencion)), [f"{d.day:02d}/{d.month:02d}" for d in retencion.index], fontsize=8)
ax.set(xlabel="semanas desde la primera visita", ylabel="cohorte (semana de la primera visita)",
       title="Retención semanal por cohorte (% de la semana 0)")
ax.grid(False)
fig.colorbar(im, ax=ax, label="% de visitantes activos", shrink=0.8)
fig.tight_layout()
plt.show()

Y la curva media, solo con las cohortes que ya tienen observada la semana
12, para que todos los puntos usen las mismas cohortes:

In [ ]:
# notna: el valor no está vacío (la cohorte ya ha llegado a la semana 12).
cohortes_12 = retencion[retencion[12].notna()]
print(f"Cohortes con 12 semanas observadas: {len(cohortes_12)} "
      f"(de la del {cohortes_12.index[0]:%d/%m} a la del {cohortes_12.index[-1]:%d/%m})")
curva_media = cohortes_12.loc[:, 1:12].mean()   # media de cada columna (semana)
print((100 * curva_media).to_string(float_format="{:.2f}".format))

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(curva_media.index, 100 * curva_media.to_numpy(), marker="o", color=AZUL,
        linewidth=2, label=f"media de las {len(cohortes_12)} cohortes con 12 semanas")
primera = retencion.iloc[0, 1:13]
ax.plot(primera.index, 100 * primera.to_numpy(), color=NARANJA, linewidth=2,
        linestyle="--", label="cohorte del 4 de mayo")
ax.set(title="Retención semanal por cohorte (semana 0 = 100 %)",
       xlabel="semanas desde la primera visita", ylabel="% de visitantes activos",
       ylim=(0, None))
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

Preguntas: (i) ¿qué proporción de visitantes vuelve la semana siguiente a su
primera visita y cuántos siguen activos a los dos meses? (ii) Las dos
primeras cohortes retienen más que las demás. ¿La tienda empeoró? (iii) La
cohorte de la última semana tiene muchos menos visitantes activos que la de
mayo. ¿Retiene peor? (iv) ¿Qué parte de la baja retención puede deberse a
que `visitor_id` es una cookie, y cómo presentarías la cifra en un informe?

**Tu respuesta:**

### B5. Transiciones entre estados

Si te has atascado antes, activa la celda de rescate: crea `eventos` sin
duplicados y con las sesiones de 30 minutos, y `por_sesion` (tarda unos
segundos).

In [ ]:
# Celda de rescate: cambia False por True solo si no has llegado hasta aquí.
USAR_RESCATE = False
if USAR_RESCATE:
    def sesionizar(eventos, umbral_min):
        """Asigna un número de sesión: empieza una nueva tras más de `umbral_min` sin actividad."""
        eventos = eventos.sort_values(["visitor_id", "timestamp"]).copy()
        hueco = eventos.groupby("visitor_id")["timestamp"].diff()
        nueva = hueco.isna() | (hueco > pd.Timedelta(minutes=umbral_min))
        eventos["session_id"] = nueva.cumsum()
        return eventos

    eventos = sesionizar(pd.read_parquet("events.parquet").drop_duplicates(), 30)
    por_sesion = pd.crosstab(eventos["session_id"], eventos["event"]).gt(0)
    print(f"Rescate cargado: {eventos['session_id'].nunique():,} sesiones; "
          f"{(por_sesion['view'] & por_sesion['addtocart']).sum():,} con vista y carrito")

El funnel impone un camino lineal: vista, carrito, compra. Pero los
visitantes no siguen caminos lineales: miran, añaden, vuelven a mirar,
añaden otra cosa, se van. Para describir esos recorridos sin imponerles una
forma, tratamos cada tipo de evento como un **estado** y contamos cuántas
veces se pasa de un estado a otro. Añadimos un estado artificial, **fin**,
para el momento en que la sesión termina.

**Ejemplo resuelto: cinco recorridos cortos**, donde V es vista, C carrito y
T compra: VVCT, VV, VC, VCT y VVV. A mano, las transiciones que salen de V:
en VVCT hay V→V y V→C; en VV, V→V y V→fin; en VC, V→C; en VCT, V→C; en VVV,
V→V, V→V y V→fin. En total, 9 transiciones desde V: 4 hacia V, 3 hacia C y
2 hacia fin. En código, cada recorrido se convierte en una lista de estados
con **fin** al final, y un bucle empareja cada estado con el siguiente;
`pd.crosstab` cuenta los pares y `reindex` ordena filas y columnas:

In [ ]:
recorridos = ["VVCT", "VV", "VC", "VCT", "VVV"]
desde = []   # estado actual de cada transición
hacia = []   # estado siguiente de cada transición
for recorrido in recorridos:
    estados = list(recorrido) + ["fin"]    # "VC" -> ["V", "C", "fin"]
    for i in range(len(estados) - 1):      # cada estado con el siguiente
        desde.append(estados[i])
        hacia.append(estados[i + 1])
# crosstab cuenta cuántas veces aparece cada par (desde, hacia).
conteos = pd.crosstab(pd.Series(desde, name="desde"), pd.Series(hacia, name="hacia"))
# reindex ordena filas y columnas; fill_value=0 para los pares que no aparecen.
conteos = conteos.reindex(index=["V", "C", "T"], columns=["V", "C", "T", "fin"], fill_value=0)
print(conteos)
print()
conteos.div(conteos.sum(axis=1), axis=0)   # cada fila entre su total

La tabla normalizada por filas es la **matriz de transición**: la fila de un
estado da la probabilidad de cada estado siguiente, estimada como una
frecuencia relativa:

$$
\hat P(\text{siguiente} = j \mid \text{actual} = i) =
\frac{\text{número de transiciones de } i \text{ a } j}
     {\text{número de transiciones que salen de } i}.
$$

Cada fila suma 1. Desde un carrito, en el ejemplo, dos de cada tres veces se
compra y una se abandona. A los puntos del recorrido donde los visitantes
muestran intención de compra y aun así se van sin comprar se les llama
**callejones sin salida**: en la matriz, la celda carrito → fin; en los
recorridos, las sesiones que pasan por el carrito y terminan sin compra. Dibujada como **diagrama de estados** (un círculo
por estado y una flecha por transición, con grosor proporcional a su
probabilidad), es la forma habitual de enseñar un journey a un equipo de
producto:

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
prob = conteos.div(conteos.sum(axis=1), axis=0)
nodos = {"V": (0, 0), "C": (2.2, 0), "T": (4.4, 0), "fin": (2.2, -2)}
fig, ax = plt.subplots(figsize=(6.5, 3.9))
for estado_dib, (x, y) in nodos.items():
    ax.add_patch(plt.Circle((x, y), 0.4, color={"fin": GRIS}.get(estado_dib, ESTILO[
        {"V": "view", "C": "addtocart", "T": "transaction"}.get(estado_dib, "view")][0]), zorder=3))
    ax.text(x, y, estado_dib, ha="center", va="center", color="white", fontsize=13,
            fontweight="bold", zorder=4)
for origen, fila in prob.iterrows():
    for destino, p in fila.items():
        if p == 0:
            continue
        etiqueta = f"{100 * p:.0f} %"
        (x0, y0), (x1, y1) = nodos[origen], nodos[destino]
        if origen == destino:
            ax.annotate("", xy=(x0 + 0.2, y0 + 0.35), xytext=(x0 - 0.2, y0 + 0.35),
                        arrowprops=dict(arrowstyle="-|>", color=GRIS, linewidth=1 + 6 * p,
                                        connectionstyle="arc3,rad=-2", shrinkA=0, shrinkB=0))
            ax.text(x0, y0 + 0.95, f"{origen}→{destino}  {etiqueta}", ha="center", fontsize=9)
            continue
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0), zorder=2,
                    arrowprops=dict(arrowstyle="-|>", color=GRIS, linewidth=1 + 6 * p,
                                    shrinkA=19, shrinkB=19))
        desplazamiento = -0.7 * np.sign(x1 - x0) if y1 != y0 else 0   # fuera del triángulo
        ax.text((x0 + x1) / 2 + desplazamiento, (y0 + y1) / 2 + 0.12, etiqueta,
                ha="center", va="bottom",
                fontsize=9, bbox=dict(facecolor="white", edgecolor="none", pad=0.4))
ax.set(xlim=(-0.8, 5.2), ylim=(-2.6, 1.3), aspect="equal")
ax.axis("off")
ax.set_title("Transiciones del ejemplo (grosor proporcional a la probabilidad)", fontsize=10)
fig.tight_layout()
plt.show()

**El supuesto de Markov y su límite.** Una matriz así es una **cadena de
Markov de primer orden**: supone que la probabilidad del siguiente paso
depende **solo del estado actual**, no de cómo se llegó a él. Es una
simplificación fuerte: un visitante que lleva veinte vistas y tres carritos
probablemente no se comporta igual que uno que acaba de añadir su primer
producto, aunque los dos estén en el estado "carrito". A cambio, la matriz
es fácil de estimar, leer y comunicar, y describe bien el comportamiento
**medio**. La usamos como herramienta **descriptiva** del journey, no para
predecir el siguiente paso de cada visitante.

**Con un log real, sin bucles.** En un log hay una fila por evento, no una
cadena por sesión. El estado siguiente de cada evento es el evento de la
fila de abajo **dentro de la misma sesión**: `groupby("session_id").shift(-1)`
lo trae, y el último evento de cada sesión, que no tiene siguiente, queda
nulo y se rellena con `"fin"` (`fillna`). El primer evento de cada sesión
(`groupby(...).first()`) da las transiciones desde un estado **inicio**, que
se apilan con las demás con `pd.concat`. Con los cinco recorridos escritos
como log (una fila por evento, ya ordenadas dentro de cada sesión):

In [ ]:
filas_ejemplo = []
for numero, recorrido in enumerate(recorridos, start=1):
    for letra_ej in recorrido:
        filas_ejemplo.append({"session_id": numero, "event": letra_ej})
log_recorridos_ejemplo = pd.DataFrame(filas_ejemplo)

estado_ejemplo = log_recorridos_ejemplo["event"]
# shift(-1) trae el evento siguiente de la misma sesión; el último no tiene: "fin".
siguiente_ejemplo = estado_ejemplo.groupby(log_recorridos_ejemplo["session_id"]).shift(-1).fillna("fin")
# Primer evento de cada sesión, para las transiciones desde "inicio".
inicio_rec_ejemplo = log_recorridos_ejemplo.groupby("session_id")["event"].first()
# pd.concat apila las dos tablas de pares (desde, hacia) una debajo de otra.
pares_ejemplo = pd.concat([
    pd.DataFrame({"desde": "inicio", "hacia": inicio_rec_ejemplo.to_numpy()}),
    pd.DataFrame({"desde": estado_ejemplo.to_numpy(), "hacia": siguiente_ejemplo.to_numpy()}),
], ignore_index=True)
# normalize="index": cada fila se divide por su total (la matriz de transición).
matriz_ejemplo = pd.crosstab(pares_ejemplo["desde"], pares_ejemplo["hacia"], normalize="index")
matriz_ejemplo.reindex(index=["inicio", "V", "C", "T"], columns=["V", "C", "T", "fin"], fill_value=0)

Las filas de V, C y T son las del bucle; la fila nueva, inicio, dice que las
cinco sesiones empiezan con una vista.

**Recorridos colapsados.** Otra forma de ver los recorridos es escribir cada
sesión como una cadena de letras **colapsando las repeticiones seguidas**
(VVVC se convierte en VC, porque lo que interesa es el cambio de estado) y
contar las cadenas más frecuentes. Una letra es una repetición si es igual a
la anterior de la misma sesión (`shift()` sin argumento trae la fila de
**arriba**); se quitan las repeticiones y `"".join` pega las letras de cada
sesión:

In [ ]:
letra_ejemplo = log_recorridos_ejemplo["event"]
# repetida: True si la letra es igual a la anterior de la misma sesión.
repetida_ejemplo = letra_ejemplo.eq(letra_ejemplo.groupby(log_recorridos_ejemplo["session_id"]).shift())
no_repetidas_ejemplo = letra_ejemplo[~repetida_ejemplo]                 # ~ invierte True/False
sesion_no_rep_ejemplo = log_recorridos_ejemplo.loc[~repetida_ejemplo, "session_id"]
# "".join pega las letras de cada sesión en una cadena: V, C -> "VC".
colapsadas_ejemplo = no_repetidas_ejemplo.groupby(sesion_no_rep_ejemplo).agg("".join)
colapsadas_ejemplo

VVCT pasa a VCT, VV a V y VVV a V: cinco sesiones, tres recorridos
distintos.

### Ejercicio 8 · La matriz de transición real · completar código + interpretar

El ejemplo de los cinco recorridos escritos como log hace lo mismo. Pistas:

```python
# pd.crosstab(desde, hacia, normalize="index") divide cada fila por su total.
# pd.concat([tabla1, tabla2], ignore_index=True) apila dos tablas (la fila "inicio" va en la primera).
```

(a) Completa `siguiente` (evento siguiente de la misma sesión, `"fin"` si no
hay), `inicio` (primer evento de cada sesión) y la matriz normalizada por
filas. La celda tarda unos segundos.

In [ ]:
# TU CÓDIGO
ordenados = eventos.sort_values(["session_id", "timestamp"])
estado = ordenados["event"]
siguiente = COMPLETA  # TODO: evento siguiente de la misma sesión, o "fin"
inicio = COMPLETA  # TODO: primer evento de cada sesión
pares_reales = pd.concat([
    pd.DataFrame({"desde": "inicio", "hacia": inicio.to_numpy()}),
    pd.DataFrame({"desde": estado.to_numpy(), "hacia": siguiente.to_numpy()}),
], ignore_index=True)
orden = ["view", "addtocart", "transaction", "fin"]
matriz = COMPLETA  # TODO: matriz normalizada por filas
matriz = matriz.reindex(index=["inicio", "view", "addtocart", "transaction"], columns=orden)
matriz

In [ ]:
# Comprobación
assert round(matriz.loc["addtocart", "fin"], 4) == 0.2847, \
    "Revisa siguiente: el último evento de cada sesión va seguido de 'fin' (fillna)."
assert round(matriz.loc["inicio", "view"], 4) == 0.9949, "Revisa inicio: el primer evento de cada sesión."
print("✔ Correcto")

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
etiquetas = {"inicio": "inicio", "view": "vista", "addtocart": "carrito",
             "transaction": "compra", "fin": "fin"}
fig, ax = plt.subplots(figsize=(6.5, 3.8))
valores_m = 100 * matriz.to_numpy(dtype=float)
ax.imshow(valores_m, cmap="Blues", vmin=0, vmax=110)
for (i, j), v in np.ndenumerate(valores_m):
    ax.text(j, i, f"{v:.1f} %".replace(".", ","), ha="center", va="center", fontsize=10,
            color="white" if v > 60 else "#0b0b0b")
fila_c, col_fin = list(matriz.index).index("addtocart"), list(matriz.columns).index("fin")
ax.add_patch(plt.Rectangle((col_fin - 0.5, fila_c - 0.5), 1, 1, fill=False,
                           edgecolor=NARANJA, linewidth=3))
ax.set_xticks(range(matriz.shape[1]), [etiquetas[c] for c in matriz.columns])
ax.set_yticks(range(matriz.shape[0]), [etiquetas[c] for c in matriz.index])
ax.xaxis.tick_top()
ax.set(xlabel="estado siguiente", ylabel="estado actual")
ax.xaxis.set_label_position("top")
ax.grid(False)
fig.tight_layout()
plt.show()

Con cuatro estados de salida y cuatro de llegada, un diagrama de estados
tendría quince flechas cruzándose: para una matriz así, el mapa de calor,
leído por filas, es más claro.

Preguntas: (i) lee la fila del carrito: ¿qué pasa después de añadir algo al
carrito? ¿Qué celda es un **callejón sin salida** (intención de compra
seguida del final de la sesión)? (ii) ¿Por qué la fila de la compra tiene un
21,5 % hacia otra compra? (iii) ¿Usarías esta matriz para decidir a qué
visitante concreto enviar un recordatorio? ¿Por qué?

**Tu respuesta:**

### Ejercicio 9 · Recorridos frecuentes y dos cifras que no coinciden · completar código + interpretar

Si no has terminado el ejercicio 8, activa la celda de rescate: crea
`ordenados`, `estado` y `siguiente`.

In [ ]:
# Celda de rescate: cambia False por True solo si no has llegado hasta aquí.
USAR_RESCATE = False
if USAR_RESCATE:
    ordenados = eventos.sort_values(["session_id", "timestamp"])
    estado = ordenados["event"]
    siguiente = estado.groupby(ordenados["session_id"]).shift(-1).fillna("fin")
    print(f"Rescate cargado: {siguiente[estado.eq('addtocart')].eq('fin').sum():,} "
          "carritos seguidos de fin")

Pista:

```python
# serie.value_counts() cuenta cuántas veces aparece cada valor, de más a menos.
```

(a) Completa la máscara `repetida`, como en el ejemplo de B5.

In [ ]:
# TU CÓDIGO
abreviatura = {"view": "V", "addtocart": "C", "transaction": "T"}
letra = ordenados["event"].map(abreviatura)   # cada evento, como su letra
repetida = COMPLETA  # TODO: True si la letra es igual a la anterior de la misma sesión
print(f"Letras repetidas: {repetida.sum():,} de {len(letra):,}")

La celda siguiente pega las letras de cada una de las 1,76 millones de
sesiones: puede tardar alrededor de un minuto.

In [ ]:
no_repetidas = letra[~repetida]
sesion_no_rep = ordenados.loc[~repetida, "session_id"]
colapsadas = no_repetidas.groupby(sesion_no_rep).agg("".join)
print(f"Sesiones: {len(colapsadas):,}")

Completa los recorridos más frecuentes y la tabla de los ocho primeros con
su porcentaje sobre todas las sesiones.

In [ ]:
# TU CÓDIGO
frecuentes = COMPLETA  # TODO: cuántas sesiones tiene cada recorrido
resumen = pd.DataFrame({
    "sesiones": frecuentes.head(8),
    "pct": COMPLETA,  # TODO: porcentaje sobre todas las sesiones
})
resumen

In [ ]:
# Comprobación
assert len(colapsadas) == 1_761_675, "colapsadas debe tener una cadena por sesión."
assert frecuentes["V"] == 1_715_386 and frecuentes["VC"] == 13_204, \
    "Revisa repetida: compara cada letra con la anterior de la misma sesión (shift())."
print("✔ Correcto")

Como V se come casi todas las sesiones, la dejamos fuera del gráfico. En
naranja, los recorridos que pasan por el carrito y no terminan en compra; en
verde, los que incluyen una compra:

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
top = frecuentes.drop("V").head(12)[::-1]
colores = [AQUA if "T" in r else NARANJA for r in top.index]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.barh(top.index, top.to_numpy(), color=colores, height=0.65)
for i, v in enumerate(top.to_numpy()):
    ax.text(v + 150, i, f"{v:,}".replace(",", " "), va="center", fontsize=9)
ax.tick_params(axis="y", labelsize=10)
solo_vistas = f"{frecuentes['V']:,}".replace(",", " ")
ax.set(title=f"Recorridos más frecuentes (sin las {solo_vistas} sesiones solo de vistas)",
       xlabel="sesiones", xlim=(0, top.max() * 1.15))
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=NARANJA),
                   plt.Rectangle((0, 0), 1, 1, color=AQUA)],
          labels=["con carrito y sin compra", "con compra"],
          frameon=False, loc="lower right", fontsize=9)
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()

(b) **Dos cifras que parecen medir lo mismo.** El *drop-off* del carrito en
el funnel por sesión es del 71,2 %; en la matriz, el 28,5 % de los carritos
va seguido de fin. La celda siguiente las concilia en tres pasos: (1) por
sesión, cuántas sesiones con carrito no compran y cuál es el último estado
de su recorrido (`.str[-1]` toma la última letra de cada cadena); (2) por
evento, qué sigue a cada evento de carrito; (3) de qué sesiones son los
carritos seguidos de fin.

In [ ]:
# Por sesión (funnel): sesiones con vista y carrito que no compran.
con_carrito = por_sesion["view"] & por_sesion["addtocart"]
sin_compra = con_carrito & ~por_sesion["transaction"]
print(f"Sesiones con carrito: {con_carrito.sum():,}; sin compra: {sin_compra.sum():,} "
      f"({sin_compra.sum() / con_carrito.sum():.1%})")
# Último estado de esas sesiones: la última letra de su recorrido colapsado (.str[-1]).
ids_sin_compra = sin_compra.index[sin_compra]
ultimo = colapsadas[ids_sin_compra].str[-1]
termina_en_c = (ultimo == "C").sum()
print(f"  terminan justo después de un carrito (último estado C): {termina_en_c:,} "
      f"({termina_en_c / con_carrito.sum():.1%} de las sesiones con carrito)")
print(f"  vuelven a mirar y terminan en una vista (último estado V): {(ultimo == 'V').sum():,}")
# Por evento (matriz): qué sigue a cada evento de carrito.
es_carrito = estado.eq("addtocart")
print(f"Eventos de carrito: {es_carrito.sum():,}; seguidos de fin: "
      f"{siguiente[es_carrito].eq('fin').sum():,} ({siguiente[es_carrito].eq('fin').mean():.1%})")
# Un carrito seguido de fin es el último evento de su sesión: ¿de qué sesiones son?
termina_en_carrito = colapsadas.str[-1] == "C"
print(f"Sesiones que terminan en un carrito: {termina_en_carrito.sum():,}")
print(f"  con vista y sin compra (las de arriba):       "
      f"{(termina_en_carrito & sin_compra).sum():,}")
print(f"  sin ninguna vista (empiezan en C):            "
      f"{(termina_en_carrito & ~por_sesion['view'] & ~por_sesion['transaction']).sum():,}")
print(f"  con compra antes del último carrito:          "
      f"{(termina_en_carrito & por_sesion['transaction']).sum():,}")

Del 71,2 % al 38,7 %: las sesiones que añaden al carrito, **vuelven a mirar**
y se van (VCV) son abandonos para el funnel, pero en la matriz su carrito va
seguido de una vista, no de fin. Del 38,7 % al 28,5 %: hay más eventos de
carrito (68 966) que sesiones con carrito (38 726), y solo el último de cada
sesión puede ir seguido de fin. Las cifras no se encadenan exactamente
porque la matriz incluye además las sesiones que empiezan en C (4 409) y las
que compraron antes del último carrito (255).

Preguntas: (i) ¿qué recorridos son los callejones sin salida más importantes
y cuántas sesiones suman? (ii) Explica a un compañero de producto, sin
fórmulas, por qué el 71,2 % y el 28,5 % no se contradicen.

**Tu respuesta:**

### B6. De la observación a la hipótesis

Ya tenemos observaciones sólidas: el 71,2 % de las sesiones con carrito no
compra (por sesión); el 28,5 % de los carritos son el último evento de la
sesión (por evento de carrito); entre el 2,6 % y el 4,8 % de los visitantes
vuelve la semana siguiente (por visitante). La tentación es saltar a la
causa: "abandonan el carrito porque el envío es caro".

**Correlación frente a causa.** El funnel dice **dónde** se van, no **por
qué**. El abandono del carrito es compatible con muchas explicaciones:

- los gastos de envío aparecen al final y sorprenden;
- el visitante usa el carrito como lista de deseos y compra otro día (quizá
  desde otro dispositivo, y lo contamos como abandono);
- está comparando precios con otras tiendas;
- el proceso de pago falla en algunos navegadores;
- simplemente no tenía intención de comprar ese día.

Cada explicación lleva a una acción distinta (bajar el envío, enviar un
recordatorio, mejorar el precio, arreglar un error técnico, no hacer nada),
y elegir la equivocada cuesta dinero. Por eso el paso siguiente a una
observación no es una conclusión, sino una **hipótesis**: una explicación
concreta y comprobable, con la acción que se deriva de ella y la métrica que
debería moverse si es cierta.

**La hipótesis en cuatro eslabones: observación → posible causa → acción →
métrica.** Ejemplo resuelto (H1), escrito como una tabla de una fila con las
columnas que usarás en el ejercicio 10:

In [ ]:
hipotesis_ejemplo = pd.DataFrame([{
    "hipotesis": "H1",
    "observacion": "El 71,2 % de las sesiones con carrito no compra (por sesión); 13 204 sesiones VC",
    "causa": "El coste total (envío incluido) no se ve hasta el pago",
    "accion": "Mostrar el coste de envío en la ficha de producto",
    "metrica": "Conversión carrito → compra (por sesión)",
    "impacto": 5,
    "esfuerzo": 1,
}])
hipotesis_ejemplo.T   # .T muestra la fila en vertical para que se lea entera

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
pasos_h = [("Observación", "71,2 % de las sesiones\ncon carrito no compran\n(por sesión)"),
           ("Posible causa", "el coste de envío\nno se ve hasta\nel pago"),
           ("Acción", "mostrar el envío\nen la ficha\nde producto"),
           ("Métrica", "conversión\ncarrito → compra\n(por sesión)")]
fig, ax = plt.subplots(figsize=(10, 2.4))
for i, (titulo, texto) in enumerate(pasos_h):
    x = i * 2.6
    ax.add_patch(plt.Rectangle((x, 0), 2.1, 1.5, facecolor="#eef4fc" if i != 1 else "#fdf0e9",
                               edgecolor=AZUL if i != 1 else NARANJA, linewidth=1.5))
    ax.text(x + 1.05, 1.25, titulo, ha="center", fontsize=10, fontweight="bold")
    ax.text(x + 1.05, 0.6, texto, ha="center", va="center", fontsize=9)
    if i < 3:
        ax.annotate("", xy=(x + 2.55, 0.75), xytext=(x + 2.15, 0.75),
                    arrowprops=dict(arrowstyle="-|>", color=GRIS, linewidth=1.5))
ax.text(2.6 + 1.05, -0.25, "la parte que no sabemos: se comprueba con un experimento",
        ha="center", fontsize=9, color=NARANJA, style="italic")
ax.set(xlim=(-0.1, 10.1), ylim=(-0.45, 1.6))
ax.axis("off")
fig.tight_layout()
plt.show()

Solo el primer eslabón sale de los datos. La causa es una conjetura (por eso
va en otro color), y la acción y la métrica se derivan de ella. Si la causa
fuera otra (por ejemplo, que el carrito se usa como lista de deseos),
cambiarían también la acción y la métrica. Fíjate en que la observación y la
métrica usan **la misma unidad** (la sesión): así, si se hace el cambio, se
puede medir con la misma cifra.

**Priorizar: impacto y esfuerzo.** Con varias hipótesis y recursos
limitados hay que elegir por dónde empezar. Una forma sencilla es puntuar
cada una en dos ejes: **impacto** (cuánto mejoraría el negocio si la
hipótesis fuera cierta: cuántas sesiones o clientes afecta y cuánto podría
moverse la métrica) y **esfuerzo** (cuánto cuesta implementar la acción y
medirla). Cada eje se puntúa de **1 (bajo) a 5 (alto)**: en el ejemplo de H1,
impacto 5 y esfuerzo 1. Tres hipótesis de ejemplo:

| # | Observación | Posible causa | Acción | Impacto | Esfuerzo | Prioridad |
| --- | --- | --- | --- | --- | --- | --- |
| H1 | 71,2 % de las sesiones con carrito no compra | El envío no se ve hasta el pago | Envío visible en la ficha | Alto: la etapa con más intención | Bajo: cambio de interfaz | 1 |
| H2 | 97,4 % de las sesiones son solo vistas | Los productos vistos no interesan | Recomendaciones de productos relacionados | Alto: casi todas las sesiones | Medio-alto: un recomendador (clases 9–12) | 2 |
| H3 | 2,6–4,8 % vuelve la semana siguiente | No hay motivo para volver | Recordatorio del carrito a quien se registra | Medio: solo registrados | Medio: identificar y pedir consentimiento | 3 |

Se colocan en una **matriz de impacto y esfuerzo**:

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
# Posiciones cualitativas, tomadas de la tabla de priorización (juicio del equipo).
hipotesis_dib = {"H1\nenvío visible": (1.0, 4.2), "H2\nrecomendaciones": (3.6, 4.4),
                 "H3\nrecordatorio": (2.8, 2.9)}
fig, ax = plt.subplots(figsize=(6, 4.6))
ax.axhline(2.5, color=GRIS, linewidth=1)
ax.axvline(2.5, color=GRIS, linewidth=1)
for (x, y), texto in {(1.25, 4.85): "hacer primero", (3.75, 4.85): "planificar",
                      (1.25, 0.3): "si sobra tiempo", (3.75, 0.3): "descartar"}.items():
    ax.text(x, y, texto, ha="center", fontsize=10, color=GRIS, style="italic")
for (nombre, (x, y)), color in zip(hipotesis_dib.items(), [AQUA, AZUL, NARANJA]):
    ax.scatter(x, y, s=240, color=color, zorder=3)
    ax.annotate(nombre, (x, y), xytext=(14, -10), textcoords="offset points", fontsize=9)
ax.set(xlim=(0, 5), ylim=(0, 5.2), xticks=[0.5, 4.5], yticks=[0.5, 4.5],
       xlabel="esfuerzo", ylabel="impacto", title="Priorización de las hipótesis")
ax.set_xticklabels(["bajo", "alto"])
ax.set_yticklabels(["bajo", "alto"])
ax.grid(False)
fig.tight_layout()
plt.show()

Impacto alto y esfuerzo bajo van primero: H1 cae sola en ese cuadrante. H2
tiene tanto impacto como H1, pero exige un proyecto, así que se planifica.
H3 queda cerca del centro. Las puntuaciones no son medidas, sino **juicios
del equipo**: su valor está en obligar a explicitarlos y discutirlos.

¿Cómo se sabe después si la hipótesis era cierta? Con un **experimento**:
mostrar el coste de envío a la mitad de los visitantes, elegidos al azar, y
comparar la conversión de las dos mitades. Es lo único que permite pasar de
correlación a causa, y es el objeto de la clase 14.

**Optimización responsable.** Optimizar la conversión es legítimo; hacerlo
engañando al cliente, no. Se llaman ***dark patterns*** (patrones oscuros)
los diseños que empujan al usuario a hacer algo que no haría si entendiera
bien lo que pasa:

- **urgencia falsa**: "¡solo quedan 2 unidades!" o cuentas atrás que se
  reinician;
- **productos añadidos por defecto**: un seguro o una suscripción marcados en
  el carrito;
- **precios por goteo**: mostrar un precio bajo y añadir cargos en el último
  paso (justo lo contrario de la acción de H1);
- **dificultar la salida**: darse de alta con un clic y de baja con una
  llamada.

Todos pueden subir la conversión a corto plazo; también erosionan la
confianza, generan devoluciones y reclamaciones y, en la Unión Europea,
varios están limitados por la normativa de consumidores y la Ley de
Servicios Digitales. Un criterio sencillo: **¿funcionaría igual si el
cliente entendiera exactamente lo que estamos haciendo?** Si la respuesta es
no, probablemente es un patrón oscuro.

### Ejercicio 10 · El journey y tres hipótesis priorizadas · completar código + abierto

Es el entregable de la clase. Pistas:

```python
# Una tabla se puede escribir fila a fila como lista de diccionarios:
# pd.DataFrame([{"columna": valor, ...}, {...}])  (como hipotesis_ejemplo).
# tabla.sort_values(["impacto", "esfuerzo"], ascending=[False, True]) ordena por dos columnas.
```

(a) El resumen cuantitativo del journey. La tabla `journey` da cada métrica
con su unidad y su umbral. Escribe en `valores_journey` las cifras que has
obtenido en los ejercicios, **como proporciones** (0,25 y no 25 %).

In [ ]:
journey = pd.DataFrame({
    "metrica": ["vista -> carrito", "carrito -> compra", "conversion total por sesion",
                "conversion total por visitante", "carritos seguidos de fin",
                "retencion semana 1"],
    "unidad": ["sesión", "sesión", "sesión con vista", "visitante con vista",
               "evento de carrito", "visitante, media de 7 cohortes con 12 semanas"],
    "umbral": ["30 min", "30 min", "30 min", "—", "30 min", "—"],
})
journey

In [ ]:
# TU CÓDIGO
valores_journey = {
    "vista -> carrito": COMPLETA,  # TODO: conversión de etapa vista -> carrito, por sesión (E4)
    "carrito -> compra": COMPLETA,  # TODO: conversión de etapa carrito -> compra, por sesión (E4)
    "conversion total por sesion": COMPLETA,  # TODO: conversión total del funnel por sesión (E4)
    "conversion total por visitante": COMPLETA,  # TODO: conversión total del funnel por visitante (E5)
    "carritos seguidos de fin": COMPLETA,  # TODO: matriz de transición, carrito -> fin (E8)
    "retencion semana 1": COMPLETA,  # TODO: curva media, semana 1 (E7)
}
journey["valor"] = journey["metrica"].map(valores_journey)
journey

In [ ]:
# Comprobación
referencia_journey = {"vista -> carrito": 0.0221, "carrito -> compra": 0.2877,
                      "conversion total por sesion": 0.0063,
                      "conversion total por visitante": 0.0073,
                      "carritos seguidos de fin": 0.2847, "retencion semana 1": 0.0372}
assert journey["valor"].notna().all(), "Falta algún valor en valores_journey."
assert journey["valor"].max() < 1, "Escribe proporciones (0,2877), no porcentajes (28,77)."
for metrica, valor in referencia_journey.items():
    mio = journey.set_index("metrica").loc[metrica, "valor"]
    assert abs(mio - valor) < 0.001, f"Revisa '{metrica}': esperaba un valor cercano a {valor}."
journey.to_csv(DIR_SALIDAS / "journey.csv", index=False)
print("✔ Correcto: guardado en salidas/journey.csv")

(b) Escribe tres hipótesis, cada una con observación (con su cifra **y su
unidad**), posible causa, acción, métrica que debería moverse, impacto y
esfuerzo (de 1 a 5). Al menos dos deben ser distintas de H1, H2 y H3. Las
celdas siguientes las ordenan por prioridad (impacto alto y esfuerzo bajo
primero), las dibujan y las guardan.

In [ ]:
# TU CÓDIGO
hipotesis = pd.DataFrame([
    {
        "hipotesis": "A",
        "observacion": COMPLETA,  # TODO: lo observado, con cifra y unidad
        "causa": COMPLETA,  # TODO: posible causa (una conjetura)
        "accion": COMPLETA,  # TODO: acción que se deriva de la causa
        "metrica": COMPLETA,  # TODO: métrica que debería moverse
        "impacto": COMPLETA,  # TODO: de 1 a 5
        "esfuerzo": COMPLETA,  # TODO: de 1 a 5
    },
    {
        "hipotesis": "B",
        "observacion": COMPLETA,  # TODO: lo observado, con cifra y unidad
        "causa": COMPLETA,  # TODO: posible causa (una conjetura)
        "accion": COMPLETA,  # TODO: acción que se deriva de la causa
        "metrica": COMPLETA,  # TODO: métrica que debería moverse
        "impacto": COMPLETA,  # TODO: de 1 a 5
        "esfuerzo": COMPLETA,  # TODO: de 1 a 5
    },
    {
        "hipotesis": "C",
        "observacion": COMPLETA,  # TODO: lo observado, con cifra y unidad
        "causa": COMPLETA,  # TODO: posible causa (una conjetura)
        "accion": COMPLETA,  # TODO: acción que se deriva de la causa
        "metrica": COMPLETA,  # TODO: métrica que debería moverse
        "impacto": COMPLETA,  # TODO: de 1 a 5
        "esfuerzo": COMPLETA,  # TODO: de 1 a 5
    },
])
hipotesis.T   # .T muestra cada hipótesis en una columna para que se lea entera

In [ ]:
# Comprobación
assert len(hipotesis) == 3, "Escribe exactamente tres hipótesis."
texto = hipotesis[["observacion", "causa", "accion", "metrica"]]
assert not texto.isin(["...", ""]).any().any(), "Completa todos los textos."
assert hipotesis["impacto"].between(1, 5).all() and hipotesis["esfuerzo"].between(1, 5).all(), \
    "Impacto y esfuerzo van de 1 a 5."
print("✔ Correcto")

In [ ]:
priorizadas = hipotesis.sort_values(["impacto", "esfuerzo"], ascending=[False, True]).copy()
priorizadas["prioridad"] = range(1, len(priorizadas) + 1)
priorizadas.to_csv(DIR_SALIDAS / "hipotesis.csv", index=False)
print("Guardado en salidas/hipotesis.csv")
priorizadas[["hipotesis", "accion", "impacto", "esfuerzo", "prioridad"]]

In [ ]:
# Código de la figura (no hace falta entenderlo para seguir la clase).
fig, ax = plt.subplots(figsize=(6, 4.6))
ax.axhline(3, color=GRIS, linewidth=1)
ax.axvline(3, color=GRIS, linewidth=1)
for (x, y), texto_cuadrante in {(1.75, 5.35): "hacer primero", (4.25, 5.35): "planificar",
                                (1.75, 0.6): "si sobra tiempo", (4.25, 0.6): "descartar"}.items():
    ax.text(x, y, texto_cuadrante, ha="center", fontsize=10, color=GRIS, style="italic")
for _, fila in priorizadas.iterrows():
    ax.scatter(fila["esfuerzo"], fila["impacto"], s=240, color=AZUL, zorder=3)
    ax.annotate(f"{fila['hipotesis']} (prioridad {fila['prioridad']})",
                (fila["esfuerzo"], fila["impacto"]), xytext=(12, -12),
                textcoords="offset points", fontsize=9)
ax.set(xlim=(0.5, 5.5), ylim=(0.5, 5.6), xticks=range(1, 6), yticks=range(1, 6),
       xlabel="esfuerzo (1 = bajo)", ylabel="impacto (5 = alto)", title="Tus hipótesis")
ax.grid(False)
fig.tight_layout()
plt.show()

Preguntas: (i) el 28,5 % de los carritos termina la sesión: propón dos
causas posibles que lleven a acciones distintas. ¿Cómo sabrías cuál es la
correcta? (ii) Alguien propone un mensaje "tu carrito caduca en 10 minutos"
para subir la conversión. ¿Lo aceptarías? Usa el criterio de B6.

**Tu respuesta:**

## Parte C · Síntesis

### Resumen de la clase

1. Agregar a nivel de cliente borra el orden, los tiempos y los recorridos.
   El *customer journey* trata al cliente como una secuencia de eventos.
2. Un event log registra quién, cuándo, qué y sobre qué. Si no trae
   sesiones, se construyen con un umbral de inactividad: ordenar, calcular
   huecos y hacer la suma acumulada de los inicios (1,76 millones de
   sesiones con 30 minutos).
3. El umbral es una decisión: se documenta y se comprueba la sensibilidad
   (entre 5 y 120 minutos, las sesiones con compra van del 0,88 % al
   0,78 %). Un buen umbral separa los huecos dentro de una visita de los
   huecos entre visitas.
4. Un funnel se lee etapa a etapa: conversión, *drop-off* y tiempo entre
   etapas. Aquí, el 2,2 % de las sesiones con vista llega al carrito y de
   ellas compra el 28,8 %; las que compran deciden en minutos (medianas de
   3,0 y 4,3).
5. El funnel por sesión (0,63 %) y por visitante (0,73 %) responden a
   preguntas distintas: nunca se mezclan, y cada cifra lleva su unidad.
6. Las cohortes comparan clientes a la misma edad, por filas, columnas y
   diagonales. La retención de la semana 1 va del 2,6 % al 4,8 %, con
   censura en las primeras cohortes y cookies que subestiman a las personas.
7. La matriz de transición describe los recorridos medios y revela
   callejones sin salida (28,5 % de los carritos seguidos de fin; 22 097
   sesiones VC o VCV); supone que solo importa el estado actual.
8. Lo observado no es la causa. Las mejoras se formulan como hipótesis
   priorizadas por impacto y esfuerzo, se validan con experimentos y nunca
   se apoyan en *dark patterns*.

### 1. De la métrica a la decisión

| Paso | En esta clase |
|---|---|
| Métrica técnica | Funnel estricto por sesión (umbral de 30 min, sobre sesiones con vista): 2,2 % vista → carrito, 28,8 % carrito → compra, 0,63 % total (71,2 % de *drop-off* del carrito); 28,5 % de los eventos de carrito terminan la sesión; retención en la semana 1 entre el 2,6 % y el 4,8 % según la cohorte |
| Interpretación | La mayor pérdida en volumen es la primera etapa, pero la más accionable es el carrito: siete de cada diez sesiones con intención de compra no compran |
| Decisión | Priorizar tres hipótesis y diseñar un experimento para la primera |
| Limitación | El funnel dice dónde, no por qué; la sesión y la retención dependen de decisiones (umbral) y de una identidad imperfecta (cookies); el dataset empieza a mitad de la vida de la tienda |
| Guardrail | Toda cifra de conversión indica unidad y umbral; ninguna acción se da por buena sin un experimento; se descartan las acciones que funcionen engañando al cliente |

Añade al guardrail una regla tuya, con una cifra del laboratorio que la
justifique:

**Tu respuesta:**

### 2. La petición del equipo de producto

Responde en cuatro líneas, con cifras y unidades, a las cuatro partes de la
petición: dónde se van, cuánto tardan en decidirse los que compran, si
vuelven y qué tres mejoras propones.

**Tu respuesta:**

### 3. Errores frecuentes

Marca los que has cometido o visto durante el laboratorio:

- [ ] No ordenar los eventos por usuario y tiempo antes de sesionizar (demo,
  ejercicio 2).
- [ ] Tratar el umbral de sesión como una verdad (ejercicio 3).
- [ ] Mezclar funnels por sesión y por usuario, por ejemplo dividir
  compradores entre sesiones (ejercicio 5).
- [ ] Leer las cohortes recientes como peores cuando solo tienen menos
  tiempo de observación (ejercicio 7).
- [ ] Ignorar la censura al inicio del periodo (ejercicio 7).
- [ ] Usar la matriz de transición como si predijera a cada visitante
  (ejercicio 8).
- [ ] Concluir la causa de un abandono a partir del funnel (ejercicio 10).

### 4. Discusión

Elige una para discutir en el aula; las otras dos quedan como tarea.

1. **Dark patterns.** ¿Dónde está la frontera entre un recordatorio útil
   ("tienes productos en el carrito") y la presión indebida ("tu carrito
   caduca en 10 minutos")? ¿Quién debería decidirlo en una empresa:
   producto, legal, el equipo de datos?
2. **Visitantes anónimos frente a clientes identificados.** Para medir la
   retención de personas y no de cookies habría que identificar a los
   visitantes (que inicien sesión, cruzar dispositivos). Eso mejora la
   medición, pero exige consentimiento y aumenta la información que se
   guarda de cada persona. ¿Compensa? ¿Para qué análisis sí y para cuáles
   no?
3. **El seguimiento del comportamiento.** Registrar cada producto que mira
   un visitante es la base de este análisis. ¿Sabe el visitante que se
   registra? ¿Cuánto tiempo debería guardarse ese detalle? La minimización
   de la clase 1 se aplica también aquí: para calcular un funnel basta con
   agregados por sesión, no hace falta conservar para siempre cada clic.

### Lo que viene

La clase 4 cambia de señal: de lo que el cliente **hace** a lo que **dice**
(reseñas), y estrena el aprendizaje supervisado. La retención de hoy vuelve
en la clase 6, donde no describiremos cuántos clientes se van, sino que
intentaremos predecir **quiénes**; y las hipótesis de hoy son justo lo que
un experimento A/B sirve para contrastar en la clase 14.

## Ampliación (opcional)

### X1 · El funnel con otros umbrales

En el ejercicio 3 viste que el umbral cambia el número de sesiones. ¿Cambia
también el funnel? Repite el funnel estricto por sesión con umbrales de 10,
30 y 120 minutos y compara las conversiones de etapa. La celda puede tardar
alrededor de un minuto (sesioniza y cuenta el log tres veces).

In [ ]:
# TU CÓDIGO
    # 1. Sesioniza con el umbral (solo las columnas necesarias).
    # 2. Una fila por sesión, True si tiene algún evento de cada tipo.
    # 3. Funnel estricto.

La conversión vista → carrito no cambia (0,0221), pero la de carrito →
compra pasa de 0,2729 con 10 minutos a 0,2877 con 30 y 0,2878 con 120: con
umbrales cortos se parten más sesiones entre el carrito y la compra (el
visitante paga 15 minutos después de añadir y ya cuenta como otra sesión).
Es otro motivo para citar siempre el umbral junto a la cifra.

### X2 · ¿Robots o grandes compradores?

En el ejercicio 1 apareció un visitante con 7 757 eventos. Quita los
visitantes con más de 100 eventos y repite el funnel por sesión. ¿Qué pasa
con la conversión? Antes de decidir, mira qué hacen.

In [ ]:
# TU CÓDIGO
# 1. Visitantes con más de 100 eventos (en el log sin duplicados).
# 2. Qué hacen: sus eventos por tipo y su parte de las compras.
# 3. Funnel estricto por sesión sin ellos.

Los 408 visitantes con más de 100 eventos son el 0,03 % de los visitantes y
el 5,76 % de los eventos, pero hacen el **28,85 % de las compras**. Sin
ellos, la conversión carrito → compra por sesión baja del 28,8 % al 25,6 %.
No son ruido que haya que quitar: son compradores intensivos
(probablemente profesionales o cuentas compartidas). Como con los atípicos
de la clase 2, antes de quitar un caso extremo hay que mirar qué hace.